# P2 — Auditoria 1 (UnB 2026.2) — PetroReconcavo S.A. (RECV3)
## Passo 3: base de dados simulada, conciliação e procedimentos de auditoria (MUS)

**Preparador:** matrícula 211059370 · **Data:** 23/10/2026 · **Revisão:** ______ (campo reservado, não preenchido)

> ⚠️ **DADOS SINTÉTICOS.** As bases geradas neste notebook **não são registros reais** da PetroReconcavo. Os únicos dados reais usados são **totais publicados** (NE 7.1 da DFP 2025, Relatório da Administração, FRE 2026 e dados abertos da CVM), indicados como "REAL" no dicionário de variáveis.

### Ferramenta utilizada (e por quê)
* **Python 3 + pandas/numpy/openpyxl**, executável no **Google Colab** (ou Anaconda/Jupyter local). O professor indicou o Colab; outras ferramentas são aceitas desde que o código seja entregue e a ferramenta explicada.
* **IA generativa:** o código foi escrito com auxílio do **Claude (Anthropic)**, a partir dos comandos do aluno; o aluno conferiu e justificou os resultados. O registro dos prompts e das conferências está em `docs/USO_DE_IA.md`.
* O notebook tem o **mesmo código** dos scripts da pasta `codigo/` (gerado por `codigo/construir_notebook.py`).

### Como foi feito ("como fez o bolo")
1. **Escolha da conta.** O imobilizado é a conta de maior materialidade: 72,2% do ativo consolidado (`planilhas/DIAG_RECV3.xlsx`, aba `Ranking_Ativo`).
2. **Dados reais.** Totais por classe e por entidade da NE 7.1, polos, produção, reservas e quadro de pessoal.
3. **SIMULAÇÃO.** Geração de 532.598 itens patrimoniais com semente fixa (211059370). Os valores foram rateados em centavos para que os totais batam com a NE 7.1.
4. **Erros inseridos.** Inserção de erros controlados, registrados num log separado que os testes não leem.
5. **Conciliação.** Base comparada com a NE 7.1 e com a CVM.
6. **Testes em 100% da população** (T01–T10).
7. **MUS (Gramling, cap. 8):** intervalo, seleção sistemática e avaliação pelo LSI.
8. **Comparação.** Só no fim o log é aberto, para comparar inseridos × detectados.
9. **Base complementar de folha e benefícios a empregados,** com testes G01–G06.

### Organização
| Seção | Conteúdo | Natureza |
|---|---|---|
| 0 | Preparação do ambiente | — |
| A + B | Dados reais (alvos) e **SIMULAÇÃO** da base do imobilizado | **SIMULAÇÃO** |
| C | Conciliação, testes, MUS e comparação | Scripts de auditoria |
| D | **SIMULAÇÃO** da base de folha + testes | **SIMULAÇÃO** + scripts |
| E | Dicionário de variáveis (gerado e conferido) | Documentação |

## 0. Preparação do ambiente
No **Colab**: rode a célula abaixo depois de trocar `URL_DO_REPOSITORIO` pelo link do GitHub. Ela clona o repositório e entra na pasta. Localmente, basta abrir o notebook na raiz `P2_PetroReconcavo`.

In [1]:
# Preparacao (Colab): clona o repositorio e entra na raiz do projeto. Localmente, nao faz nada.
import os, sys
URL_DO_REPOSITORIO = 'https://github.com/jvqueirogaa/P2_PetroReconcavo.git'   # repositorio da entrega
if 'google.colab' in sys.modules and not os.path.exists('codigo'):
    os.system(f'git clone {URL_DO_REPOSITORIO} repo')
    os.chdir('repo')
    os.system('pip install -q openpyxl')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')          # executado de dentro de notebooks/: sobe para a raiz
print('pasta de trabalho:', os.getcwd())

pasta de trabalho: /sessions/rcw-01f2osvla8pvzdeuqw4aot56/mnt/Auditoria 1/P2_PetroReconcavo


## A + B. Dados reais (alvos) e SIMULAÇÃO da base do imobilizado
> 🟨 **SEÇÃO DE SIMULAÇÃO**: tudo que é gerado aqui é sintético.

Código-fonte: `codigo/gerar_base_imobilizado.py` (o mesmo código, dividido em células).

In [2]:
# -*- coding: utf-8 -*-
# =============================================================================
# P2 - AUDITORIA 1 (UnB 2026.2) - PetroReconcavo S.A. (RECV3) - exercicio 2025
# Preparador: matricula 211059370
#
# >>> SECAO DE SIMULACAO <<<
# Gera a BASE SIMULADA (DADOS SINTETICOS) do subrazao analitico do IMOBILIZADO,
# conta de maior materialidade do ativo (72,2% do ativo consolidado - ver
# planilhas/DIAG_RECV3.xlsx, aba Ranking_Ativo).
#
# !!! OS DADOS GERADOS SAO SINTETICOS. NAO SAO REGISTROS REAIS DA COMPANHIA. !!!
# Os unicos dados REAIS usados sao os TOTAIS publicados na NE 7.1 da DFP 2025
# (F01 p.68-70), os totais operacionais do Relatorio da Administracao (producao
# e reservas por ativo, F01 p.5-8) e a lista publica de campos dos polos
# (FRE 2026 item 1.1, F13 p.5-8). Todo o detalhe analitico (itens, pocos, datas,
# fornecedores, usuarios, documentos) e simulado.
#
# Uso:  python codigo/gerar_base_imobilizado.py      (na raiz P2_PetroReconcavo)
# Reprodutivel: semente fixa = matricula (211059370).
# =============================================================================
import os, json, hashlib
import numpy as np
import pandas as pd

SEED = 211059370
rng = np.random.default_rng(SEED)
DATA_BASE = pd.Timestamp('2025-12-31')          # data do balanco
OUT = 'base_simulada'
CTRL = 'base_simulada/controle_erros'           # log dos erros inseridos (NAO usado pelos testes)
os.makedirs(OUT, exist_ok=True); os.makedirs(CTRL, exist_ok=True)

In [3]:
# -----------------------------------------------------------------------------
# 1) DADOS REAIS - alvos da NE 7.1 (R$ mil). Fonte: F01 p.68 (controladora) e
#    p.69 (consolidado). SPE Tieta = consolidado - controladora (inclui a
#    mais-valia de aquisicao alocada no consolidado).
#    (custo 31/12/2025, deprec/deplecao acumulada 31/12/2025, deprec 2025 = coluna "Adicoes")
# -----------------------------------------------------------------------------
NE7_CONTROLADORA = {
    'MAQUINAS_EQUIPAMENTOS': (716487, 103733, 38872),
    'IMOBILIZADO_EM_ANDAMENTO': (68419, 0, 0),
    'DIREITO_PRODUCAO_OLEO_GAS': (2851955, 850980, 131327),
    'DESENVOLVIMENTO_CAMPOS': (4018461, 1749489, 306122),
    'BLOCOS_EXPLORATORIOS': (9553, 0, 0),
    'ABANDONO_POCO_ARO': (73827, 36676, 1156),
    'ALMOXARIFADO_INVERSOES_FIXAS': (435242, 0, 0),
    'ADIANTAMENTOS': (15677, 0, 0),
    'OUTROS': (195997, 39032, 13895),
}
NE7_CONSOLIDADO = {
    'MAQUINAS_EQUIPAMENTOS': (718521, 103788, 38884),
    'IMOBILIZADO_EM_ANDAMENTO': (72171, 0, 0),
    'DIREITO_PRODUCAO_OLEO_GAS': (2931329, 922414, 132263),
    'DESENVOLVIMENTO_CAMPOS': (5186676, 2422704, 485386),
    'BLOCOS_EXPLORATORIOS': (20157, 0, 0),
    'ABANDONO_POCO_ARO': (79303, 41848, 1174),
    'ALMOXARIFADO_INVERSOES_FIXAS': (477651, 0, 0),
    'ADIANTAMENTOS': (18419, 0, 0),
    'OUTROS': (199325, 41957, 14161),
}
ENT_PR, ENT_TI = 'PETRORECONCAVO', 'SPE_TIETA'
ALVO = {ENT_PR: NE7_CONTROLADORA,
        ENT_TI: {k: tuple(NE7_CONSOLIDADO[k][i] - NE7_CONTROLADORA[k][i] for i in range(3)) for k in NE7_CONSOLIDADO}}
# Direito de producao por polo (NE 7.1 nota ii, F01 p.70) - custo, R$ mil
UPGN_CUSTO_MIL, UPGN_DEPREC_MIL = 284999, 4322     # NE 7.1, coluna 'UPGN Guamare' (maquinas): custo e depreciacao (F01 p.68-69)
DIREITO_POR_POLO = {'REMANSO': 95629, 'REMANSO_BT_REC': 1248, 'MIRANGA': 1247506, 'POTIGUAR': 1507572, 'TIETA': 79374}
# Totais operacionais reais (F01 RA p.5-8): producao media 2025 (boe/d) e reservas 31/12/2025 (MMboe)
PROD_BOED = {'BAHIA': 13400, 'POTIGUAR': 13100}
RES_1P = {'BAHIA': 87.3, 'POTIGUAR': 56.1}
RES_PDP_TOTAL = 61.4

In [4]:
# -----------------------------------------------------------------------------
# 2) SIMULACAO - cadastro de campos (58 concessoes, F01 NE 1) e pocos
#    Nomes reais quando publicados (FRE 2026 item 1.1 e NE 7.4); demais = codigo sintetico.
# -----------------------------------------------------------------------------
campos = []
def add(polo, ent, ativo, uf, nomes, data_aq):
    for n in nomes:
        campos.append(dict(campo=n, polo=polo, entidade=ent, ativo=ativo, uf=uf, data_aquisicao_polo=data_aq))
add('REMANSO', ENT_PR, 'BAHIA', 'BA', ['BREJINHO', 'CANABRAVA', 'CASSARONGONGO', 'FAZENDA BELEM', 'GOMO', 'MATA DE SAO JOAO',
    'NORTE FAZENDA CARUACU', 'REMANSO', 'RIO DOS OVOS', 'RIO SUBAUMA', 'SAO PEDRO', 'SESMARIA'], '2021-12-23')
add('MIRANGA', ENT_PR, 'BAHIA', 'BA', ['APRAIUS', 'BIRIBA', 'FAZENDA ONCA', 'JACUIPE', 'MIRANGA', 'MIRANGA NORTE', 'RIO PIPIRI',
    'RIACHO SAO PEDRO', 'SUSSUARANA'], '2021-12-07')
add('REMANSO_BT_REC', ENT_PR, 'BAHIA', 'BA', ['ACAJA-BURIZINHO', 'LAGOA DO PAULO NORTE', 'LAGOA DO PAULO', 'LAGOA DO PAULO SUL', 'JURITI'], '2003-06-01')
add('TIETA', ENT_TI, 'BAHIA', 'BA', ['TIE'], '2023-02-28')
add('TIETA', ENT_TI, 'BAHIA', 'SE', ['TARTARUGA'], '2023-02-28')
pot_reais = ['RIACHO DA FORQUILHA', 'SABIA', 'LIVRAMENTO', 'ACAUA', 'BAIXA DO ALGODAO', 'FAZENDA CURRAL', 'FAZENDA MALAQUIAS',
             'PAJEU', 'RIO MOSSORO', 'TRES MARIAS']
add('POTIGUAR', ENT_PR, 'POTIGUAR', 'RN', pot_reais + [f'POT-CONC-{i:02d} (SINTETICO)' for i in range(1, 21)], '2019-12-09')
campos = pd.DataFrame(campos)
assert len(campos) == 58
campos['sigla'] = [f'C{i:02d}' for i in range(1, 59)]
POCOS_POLO = {'POTIGUAR': 3500, 'REMANSO': 1550, 'MIRANGA': 1600, 'REMANSO_BT_REC': 400, 'TIETA': 150}
pesos = rng.lognormal(0, 0.8, len(campos))
campos['n_pocos'] = 0
for polo, n in POCOS_POLO.items():
    m = campos.polo == polo
    w = pesos[m.values] / pesos[m.values].sum()
    q = np.maximum(3, np.floor(w * n)).astype(int); q[np.argmax(q)] += n - q.sum()
    campos.loc[m, 'n_pocos'] = q
# producao e reservas por campo (rateio SINTETICO dos totais REAIS por ativo)
for ativo in ('BAHIA', 'POTIGUAR'):
    m = campos.ativo == ativo
    w = campos.loc[m, 'n_pocos'].values * rng.lognormal(0, 0.35, m.sum()); w = w / w.sum()
    campos.loc[m, 'producao_2025_boe'] = np.round(w * PROD_BOED[ativo] * 365)
    w2 = w * rng.lognormal(0, 0.25, m.sum()); w2 = w2 / w2.sum()
    campos.loc[m, 'reservas_1P_31122025_boe'] = np.round(w2 * RES_1P[ativo] * 1e6)
campos['reservas_PDP_31122025_boe'] = np.round(campos.reservas_1P_31122025_boe * RES_PDP_TOTAL / sum(RES_1P.values()) * rng.uniform(0.85, 1.15, 58))
campos['taxa_uop_1P_2025'] = campos.producao_2025_boe / (campos.reservas_1P_31122025_boe + campos.producao_2025_boe)
campos['taxa_uop_PDP_2025'] = campos.producao_2025_boe / (campos.reservas_PDP_31122025_boe + campos.producao_2025_boe)
campos['situacao_concessao'] = 'PRODUCAO'
campos['natureza_dado'] = 'NOMES/POLOS: publicos (F13, F01); quantidades por campo: SINTETICAS (rateio de totais reais por ativo)'

pocos = campos.loc[campos.index.repeat(campos.n_pocos), ['campo', 'sigla', 'polo', 'entidade', 'ativo', 'uf', 'data_aquisicao_polo']].reset_index(drop=True)
pocos['seq'] = pocos.groupby('campo').cumcount() + 1
pocos['poco_id'] = 'PRS-' + pocos.sigla + '-' + pocos.seq.astype(str).str.zfill(4)
# data de entrada do poco no grupo: na aquisicao do polo (pocos adquiridos) ou perfurado depois
aq = pd.to_datetime(pocos.data_aquisicao_polo)
novo = rng.random(len(pocos)) < 0.22
span = (DATA_BASE - aq).dt.days.values
pocos['data_entrada'] = aq + pd.to_timedelta(np.where(novo, (rng.beta(2.2, 1.2, len(pocos)) * span).astype(int), 0), unit='D')
pocos['origem_poco'] = np.where(novo, 'PERFURADO_PELO_GRUPO', 'ADQUIRIDO_COM_O_POLO')

In [5]:
# -----------------------------------------------------------------------------
# 3) SIMULACAO - plano de contas analitico do imobilizado (SINTETICO; a DFP nao divulga plano de contas)
# -----------------------------------------------------------------------------
PLANO = [
 ('1.2.03.01.001', 'Maquinas e equip. - sondas e equipamentos de perfuracao/producao', 'MAQUINAS_EQUIPAMENTOS', '1.2.03.91.001'),
 ('1.2.03.01.002', 'Maquinas e equip. - unidades de bombeio e elevacao artificial', 'MAQUINAS_EQUIPAMENTOS', '1.2.03.91.001'),
 ('1.2.03.01.003', 'Maquinas e equip. - compressores, separadores, tanques e facilities', 'MAQUINAS_EQUIPAMENTOS', '1.2.03.91.001'),
 ('1.2.03.01.004', 'Maquinas e equip. - UPGN Guamare (50% JOA Brava)', 'MAQUINAS_EQUIPAMENTOS', '1.2.03.91.001'),
 ('1.2.03.01.005', 'Maquinas e equip. - veiculos pesados e ferramentas', 'MAQUINAS_EQUIPAMENTOS', '1.2.03.91.001'),
 ('1.2.03.02.001', 'Imobilizado em andamento - projetos de facilities', 'IMOBILIZADO_EM_ANDAMENTO', ''),
 ('1.2.03.02.002', 'Imobilizado em andamento - pocos em perfuracao/completacao', 'IMOBILIZADO_EM_ANDAMENTO', ''),
 ('1.2.03.03.001', 'Direito de producao - Polo Remanso', 'DIREITO_PRODUCAO_OLEO_GAS', '1.2.03.93.001'),
 ('1.2.03.03.002', 'Direito de producao - Remanso BT-REC', 'DIREITO_PRODUCAO_OLEO_GAS', '1.2.03.93.001'),
 ('1.2.03.03.003', 'Direito de producao - Polo Miranga', 'DIREITO_PRODUCAO_OLEO_GAS', '1.2.03.93.001'),
 ('1.2.03.03.004', 'Direito de producao - Polo Potiguar', 'DIREITO_PRODUCAO_OLEO_GAS', '1.2.03.93.001'),
 ('1.2.03.03.005', 'Direito de producao - Polo Tieta (SPE Tieta)', 'DIREITO_PRODUCAO_OLEO_GAS', '1.2.03.93.001'),
 ('1.2.03.04.001', 'Desenvolvimento de campos - perfuracao e completacao', 'DESENVOLVIMENTO_CAMPOS', '1.2.03.94.001'),
 ('1.2.03.04.002', 'Desenvolvimento de campos - workover/recompletacao capitalizados', 'DESENVOLVIMENTO_CAMPOS', '1.2.03.94.001'),
 ('1.2.03.04.003', 'Desenvolvimento de campos - equipamentos de subsuperficie e cabeca de poco', 'DESENVOLVIMENTO_CAMPOS', '1.2.03.94.001'),
 ('1.2.03.04.004', 'Desenvolvimento de campos - linhas e instalacoes de superficie', 'DESENVOLVIMENTO_CAMPOS', '1.2.03.94.001'),
 ('1.2.03.04.005', 'Desenvolvimento de campos - pocos injetores e recuperacao secundaria', 'DESENVOLVIMENTO_CAMPOS', '1.2.03.94.001'),
 ('1.2.03.05.001', 'Blocos exploratorios', 'BLOCOS_EXPLORATORIOS', ''),
 ('1.2.03.06.001', 'Abandono de pocos - ativo de desmantelamento (ARO)', 'ABANDONO_POCO_ARO', '1.2.03.96.001'),
 ('1.2.03.07.001', 'Almoxarifado p/ inversoes fixas - tubulares (OCTG)', 'ALMOXARIFADO_INVERSOES_FIXAS', ''),
 ('1.2.03.07.002', 'Almoxarifado p/ inversoes fixas - hastes, bombas e componentes de elevacao', 'ALMOXARIFADO_INVERSOES_FIXAS', ''),
 ('1.2.03.07.003', 'Almoxarifado p/ inversoes fixas - equipamentos de superficie e valvulas', 'ALMOXARIFADO_INVERSOES_FIXAS', ''),
 ('1.2.03.07.004', 'Almoxarifado p/ inversoes fixas - materiais diversos', 'ALMOXARIFADO_INVERSOES_FIXAS', ''),
 ('1.2.03.08.001', 'Adiantamentos a fornecedores de imobilizado', 'ADIANTAMENTOS', ''),
 ('1.2.03.09.001', 'Outros - terrenos', 'OUTROS', ''),
 ('1.2.03.09.002', 'Outros - edificacoes e benfeitorias', 'OUTROS', '1.2.03.99.001'),
 ('1.2.03.09.003', 'Outros - moveis e utensilios', 'OUTROS', '1.2.03.99.001'),
 ('1.2.03.09.004', 'Outros - equipamentos de informatica', 'OUTROS', '1.2.03.99.001'),
 ('1.2.03.09.005', 'Outros - veiculos leves', 'OUTROS', '1.2.03.99.001'),
]
plano = pd.DataFrame(PLANO, columns=['conta_custo', 'descricao_conta', 'classe_ne7', 'conta_depreciacao'])
plano['natureza_dado'] = 'SINTETICO - plano de contas simulado; mapeado as classes reais da NE 7.1'
DESC_DEP = {'1.2.03.91.001': '(-) Depreciacao acumulada - maquinas e equipamentos', '1.2.03.93.001': '(-) Deplecao acumulada - direito de producao',
            '1.2.03.94.001': '(-) Deplecao acumulada - desenvolvimento de campos', '1.2.03.96.001': '(-) Deplecao acumulada - abandono de pocos',
            '1.2.03.99.001': '(-) Depreciacao acumulada - outros'}

In [6]:
# -----------------------------------------------------------------------------
# 4) SIMULACAO - geracao dos itens por classe
# -----------------------------------------------------------------------------
partes = []
def bloco(n, **cols):
    d = pd.DataFrame({k: (v if np.ndim(v) else np.repeat(v, n)) for k, v in cols.items()})
    partes.append(d); return d

def datas_unif(ini, fim, n, a=1.0, b=1.0):
    ini, fim = pd.Timestamp(ini), pd.Timestamp(fim)
    return ini + pd.to_timedelta((rng.beta(a, b, n) * (fim - ini).days).astype(int), unit='D')

# 4.1 Direito de producao: 1 registro por concessao + parcelas de ajuste de preco (earnout/ajustes)
for _, c in campos.iterrows():
    k = 1 + rng.poisson(0.4)
    bloco(k, entidade=c.entidade, polo=c.polo, campo=c.campo, uf=c.uf, poco_id='', classe_ne7='DIREITO_PRODUCAO_OLEO_GAS',
          conta_custo={'REMANSO': '1.2.03.03.001', 'REMANSO_BT_REC': '1.2.03.03.002', 'MIRANGA': '1.2.03.03.003', 'POTIGUAR': '1.2.03.03.004', 'TIETA': '1.2.03.03.005'}[c.polo],
          tipo_item=['CUSTO DE AQUISICAO DA CONCESSAO'] + ['AJUSTE DE PRECO DE AQUISICAO'] * (k - 1),
          origem={'TIETA': 'AQUISICAO_MAHA', 'REMANSO_BT_REC': 'RODADA_ANP'}.get(c.polo, 'AQUISICAO_PETROBRAS'),
          data_aquisicao=pd.Timestamp(c.data_aquisicao_polo) + pd.to_timedelta(np.r_[0, rng.integers(30, 700, k - 1)], unit='D'),
          metodo_depreciacao='UOP_1P', taxa_anual_linear=np.nan, peso=np.r_[1.0, rng.uniform(0.01, 0.05, k - 1)] * c.n_pocos * rng.uniform(0.7, 1.3))

# 4.2 Desenvolvimento de campos: componentes capitalizados por poco (perfuracao, completacao, workover, etc.)
TIPOS_DES = [('PERFURACAO E COMPLETACAO', '1.2.03.04.001', 9.0, 0.10), ('WORKOVER / RECOMPLETACAO CAPITALIZADO', '1.2.03.04.002', 6.0, 0.30),
             ('FRATURAMENTO / ESTIMULACAO CAPITALIZADA', '1.2.03.04.002', 6.3, 0.08), ('COLUNA DE PRODUCAO E HASTES', '1.2.03.04.003', 4.6, 0.20),
             ('BOMBA DE FUNDO (BM/BCP)', '1.2.03.04.003', 4.3, 0.12), ('CABECA DE POCO / ARVORE DE NATAL', '1.2.03.04.003', 4.4, 0.06),
             ('LINHA DE SURGENCIA E MANIFOLD', '1.2.03.04.004', 4.8, 0.09), ('CONVERSAO EM POCO INJETOR', '1.2.03.04.005', 5.9, 0.05)]
n_comp = 1 + rng.poisson(25.4, len(pocos))
idx = np.repeat(np.arange(len(pocos)), n_comp)
P = pocos.iloc[idx].reset_index(drop=True)
probs = np.array([t[3] for t in TIPOS_DES]); probs = probs / probs.sum()
tp = rng.choice(len(TIPOS_DES), len(P), p=probs)
ini = np.maximum(P.data_entrada.values.astype('datetime64[D]'), np.datetime64('2000-02-01'))
dias = (np.datetime64('2025-12-31') - ini).astype(int)
dt_des = ini + (rng.beta(1.6, 1.0, len(P)) * dias).astype('timedelta64[D]')
mu = np.array([TIPOS_DES[t][2] for t in tp])
bloco(len(P), entidade=P.entidade.values, polo=P.polo.values, campo=P.campo.values, uf=P.uf.values, poco_id=P.poco_id.values,
      classe_ne7='DESENVOLVIMENTO_CAMPOS', conta_custo=np.array([TIPOS_DES[t][1] for t in tp]), tipo_item=np.array([TIPOS_DES[t][0] for t in tp]),
      origem='CAPEX_PROPRIO', data_aquisicao=pd.to_datetime(dt_des), metodo_depreciacao='UOP_PDP', taxa_anual_linear=np.nan,
      peso=rng.lognormal(mu, 1.05))

# 4.3 Abandono de poco (ativo ARO): um registro por poco
bloco(len(pocos), entidade=pocos.entidade.values, polo=pocos.polo.values, campo=pocos.campo.values, uf=pocos.uf.values, poco_id=pocos.poco_id.values,
      classe_ne7='ABANDONO_POCO_ARO', conta_custo='1.2.03.06.001', tipo_item='CUSTO ESTIMADO DE ABANDONO (VALOR PRESENTE)', origem='ARO_REMENSURACAO',
      data_aquisicao=pocos.data_entrada.values, metodo_depreciacao='UOP_PDP', taxa_anual_linear=np.nan, peso=rng.lognormal(0, 0.45, len(pocos)))

# 4.4 Maquinas e equipamentos (depreciacao linear 10% a.a. - NE 7.2)
def maq(n, conta, tipo, mu, sd, d0, d1, ent=ENT_PR, origem='CAPEX_PROPRIO', polo=None):
    cc = campos[campos.entidade == ent].sample(n, replace=True, weights='n_pocos', random_state=int(rng.integers(1e9)))
    bloco(n, entidade=ent, polo=cc.polo.values if polo is None else polo, campo=cc.campo.values if polo is None else 'UPGN GUAMARE', uf=cc.uf.values if polo is None else 'RN',
          poco_id='', classe_ne7='MAQUINAS_EQUIPAMENTOS', conta_custo=conta, tipo_item=tipo, origem=origem,
          data_aquisicao=datas_unif(d0, d1, n, 1.4, 1.0), metodo_depreciacao='LINEAR', taxa_anual_linear=0.10, peso=rng.lognormal(mu, sd, n))
maq(2200, '1.2.03.01.001', 'COMPONENTE DE SONDA PROPRIA (PERFURACAO/PRODUCAO)', 11.2, 1.1, '2008-01-01', '2025-11-30')
maq(5200, '1.2.03.01.002', 'UNIDADE DE BOMBEIO MECANICO', 10.6, 0.5, '2009-01-01', '2025-11-30')
maq(8400, '1.2.03.01.003', 'COMPRESSOR/SEPARADOR/TANQUE/FACILITY', 10.4, 1.0, '2008-01-01', '2025-11-30')
maq(26000, '1.2.03.01.005', 'FERRAMENTA/EQUIPAMENTO DE CAMPO', 8.2, 1.0, '2012-01-01', '2025-11-30')
maq(850, '1.2.03.01.005', 'VEICULO PESADO / CAMINHAO SONDA', 11.0, 0.6, '2012-01-01', '2025-11-30')
maq(12, '1.2.03.01.001', 'EQUIPAMENTO DE CAMPO - SPE TIETA', 10.5, 0.8, '2023-03-01', '2025-11-30', ent=ENT_TI)
n_upgn = 3100
bloco(n_upgn, entidade=ENT_PR, polo='MIDSTREAM_RN', campo='UPGN GUAMARE', uf='RN', poco_id='', classe_ne7='MAQUINAS_EQUIPAMENTOS', conta_custo='1.2.03.01.004',
      tipo_item=rng.choice(['MODULO DE PROCESSAMENTO UPGN III', 'COMPRESSAO E RECEBIMENTO', 'ARMAZENAMENTO DE LIQUIDOS', 'GASODUTO DE INTERLIGACAO (TRECHO)',
                            'UTILIDADES E INSTRUMENTACAO', 'UPGN II (HIBERNADA)'], n_upgn, p=[.30, .18, .12, .15, .20, .05]),
      origem='AQUISICAO_BRAVA_JOA', data_aquisicao=pd.Timestamp('2025-09-30'), metodo_depreciacao='LINEAR', taxa_anual_linear=0.10, peso=rng.lognormal(11.0, 1.2, n_upgn))

# 4.5 Imobilizado em andamento
for ent, n in ((ENT_PR, 2350), (ENT_TI, 150)):
    cc = campos[campos.entidade == ent].sample(n, replace=True, weights='n_pocos', random_state=int(rng.integers(1e9)))
    bloco(n, entidade=ent, polo=cc.polo.values, campo=cc.campo.values, uf=cc.uf.values, poco_id='', classe_ne7='IMOBILIZADO_EM_ANDAMENTO',
          conta_custo=rng.choice(['1.2.03.02.001', '1.2.03.02.002'], n, p=[.55, .45]), tipo_item='ORDEM DE INVESTIMENTO EM ANDAMENTO', origem='CAPEX_PROPRIO',
          data_aquisicao=datas_unif('2024-07-01', '2025-12-20', n, 2.0, 1.0), metodo_depreciacao='NAO_DEPRECIA', taxa_anual_linear=np.nan, peso=rng.lognormal(9.6, 1.3, n))

# 4.6 Blocos exploratorios (NE 17.2) - 6 registros
for ent, polo, campo, uf, d in ((ENT_PR, 'POTIGUAR', 'BLOCO POT-T-702', 'RN', '2021-03-15'), (ENT_PR, 'POTIGUAR', 'BLOCO POT-T-742', 'RN', '2024-06-27'),
                                (ENT_PR, 'POTIGUAR', 'BLOCO POT-T-793', 'RN', '2024-06-27'), (ENT_TI, 'TIETA', 'BLOCO REC-T-129', 'BA', '2023-02-28'),
                                (ENT_TI, 'TIETA', 'BLOCO REC-T-142', 'BA', '2023-02-28'), (ENT_TI, 'TIETA', 'BLOCO REC-T-155', 'BA', '2023-02-28')):
    bloco(1, entidade=ent, polo=polo, campo=campo, uf=uf, poco_id='', classe_ne7='BLOCOS_EXPLORATORIOS', conta_custo='1.2.03.05.001',
          tipo_item='BONUS DE ASSINATURA E GASTOS EXPLORATORIOS', origem='RODADA_ANP' if ent == ENT_PR else 'AQUISICAO_MAHA',
          data_aquisicao=pd.Timestamp(d), metodo_depreciacao='NAO_DEPRECIA', taxa_anual_linear=np.nan, peso=rng.uniform(0.5, 1.5, 1))

# 4.7 Almoxarifado para inversoes fixas (lotes por SKU e deposito) - nao deprecia
TIPOS_AL = [('TUBO DE REVESTIMENTO/PRODUCAO (LOTE)', '1.2.03.07.001', 9.4, .26), ('HASTE DE BOMBEIO (LOTE)', '1.2.03.07.002', 8.6, .22),
            ('BOMBA DE FUNDO EM ESTOQUE', '1.2.03.07.002', 9.0, .12), ('VALVULA/EQUIPAMENTO DE SUPERFICIE', '1.2.03.07.003', 8.4, .18),
            ('MATERIAL DIVERSO DE INVESTIMENTO', '1.2.03.07.004', 7.2, .22)]
for ent, n in ((ENT_PR, 196000), (ENT_TI, 14000)):
    cc = campos[campos.entidade == ent].sample(n, replace=True, weights='n_pocos', random_state=int(rng.integers(1e9)))
    pr = np.array([t[3] for t in TIPOS_AL]); t = rng.choice(len(TIPOS_AL), n, p=pr / pr.sum())
    bloco(n, entidade=ent, polo=cc.polo.values, campo=cc.campo.values, uf=cc.uf.values, poco_id='', classe_ne7='ALMOXARIFADO_INVERSOES_FIXAS',
          conta_custo=np.array([TIPOS_AL[i][1] for i in t]), tipo_item=np.array([TIPOS_AL[i][0] for i in t]), origem='COMPRA_MATERIAL',
          data_aquisicao=datas_unif('2021-02-01', '2025-12-20', n, 2.6, 1.0), metodo_depreciacao='NAO_DEPRECIA', taxa_anual_linear=np.nan,
          peso=rng.lognormal(np.array([TIPOS_AL[i][2] for i in t]), 1.0))

# 4.8 Adiantamentos a fornecedores de imobilizado
for ent, n in ((ENT_PR, 1650), (ENT_TI, 150)):
    cc = campos[campos.entidade == ent].sample(n, replace=True, weights='n_pocos', random_state=int(rng.integers(1e9)))
    bloco(n, entidade=ent, polo=cc.polo.values, campo=cc.campo.values, uf=cc.uf.values, poco_id='', classe_ne7='ADIANTAMENTOS', conta_custo='1.2.03.08.001',
          tipo_item='ADIANTAMENTO CONTRATUAL A FORNECEDOR', origem='CAPEX_PROPRIO', data_aquisicao=datas_unif('2025-05-01', '2025-12-28', n, 2.0, 1.0),
          metodo_depreciacao='NAO_DEPRECIA', taxa_anual_linear=np.nan, peso=rng.lognormal(8.9, 1.1, n))

# 4.9 Outros (terrenos, edificacoes, moveis, TI, veiculos leves)
TIPOS_OU = [('TERRENO', '1.2.03.09.001', 11.5, .003, 0.0), ('EDIFICACAO / BENFEITORIA', '1.2.03.09.002', 10.4, .06, 0.04),
            ('MOVEL E UTENSILIO', '1.2.03.09.003', 7.0, .45, 0.10), ('EQUIPAMENTO DE INFORMATICA', '1.2.03.09.004', 7.6, .43, 0.20),
            ('VEICULO LEVE', '1.2.03.09.005', 11.2, .057, 0.20)]
for ent, n in ((ENT_PR, 74000), (ENT_TI, 1200)):
    cc = campos[campos.entidade == ent].sample(n, replace=True, weights='n_pocos', random_state=int(rng.integers(1e9)))
    pr = np.array([t[3] for t in TIPOS_OU]); t = rng.choice(len(TIPOS_OU), n, p=pr / pr.sum())
    tx = np.array([TIPOS_OU[i][4] for i in t])
    bloco(n, entidade=ent, polo=cc.polo.values, campo=cc.campo.values, uf=cc.uf.values, poco_id='', classe_ne7='OUTROS',
          conta_custo=np.array([TIPOS_OU[i][1] for i in t]), tipo_item=np.array([TIPOS_OU[i][0] for i in t]), origem='CAPEX_PROPRIO',
          data_aquisicao=datas_unif('2006-01-01', '2025-11-30', n, 1.8, 1.0), metodo_depreciacao=np.where(tx > 0, 'LINEAR', 'NAO_DEPRECIA'),
          taxa_anual_linear=np.where(tx > 0, tx, np.nan), peso=rng.lognormal(np.array([TIPOS_OU[i][2] for i in t]), 0.9))

base = pd.concat(partes, ignore_index=True)
base['data_aquisicao'] = pd.to_datetime(base.data_aquisicao)

In [7]:
# -----------------------------------------------------------------------------
# 5) SIMULACAO - atributos de controle (cadastro, aprovacao, documentos, fornecedores)
# -----------------------------------------------------------------------------
N = len(base)
usuarios = [f'USR{i:03d}' for i in range(1, 29)]
base['usuario_cadastro'] = rng.choice(usuarios, N)
off = rng.integers(1, len(usuarios), N)
base['usuario_aprovacao'] = [usuarios[(usuarios.index(u) + o) % len(usuarios)] for u, o in zip(base.usuario_cadastro, off)]   # sempre != maker
base['data_cadastro_sistema'] = base.data_aquisicao + pd.to_timedelta(rng.integers(0, 25, N), unit='D')
base['data_cadastro_sistema'] = base.data_cadastro_sistema.where(base.data_cadastro_sistema <= DATA_BASE, DATA_BASE)
base['data_ultima_alteracao'] = base.data_cadastro_sistema + pd.to_timedelta((rng.random(N) < 0.3) * rng.integers(0, 400, N), unit='D')
base['data_ultima_alteracao'] = base.data_ultima_alteracao.where(base.data_ultima_alteracao <= DATA_BASE, DATA_BASE)
pref = base.classe_ne7.map({'ALMOXARIFADO_INVERSOES_FIXAS': 'NF', 'ADIANTAMENTOS': 'CTR', 'DIREITO_PRODUCAO_OLEO_GAS': 'CCV',
                            'ABANDONO_POCO_ARO': 'MEM', 'BLOCOS_EXPLORATORIOS': 'ANP'}).fillna('OI')
base['documento_origem'] = pref + '-' + pd.Series(rng.permutation(np.arange(1_000_000, 1_000_000 + N))).astype(str).values
base['fornecedor_id'] = 'FORN-' + pd.Series(rng.zipf(1.6, N) % 4800 + 1).astype(str).str.zfill(5).values
base.loc[base.classe_ne7 == 'DIREITO_PRODUCAO_OLEO_GAS', 'fornecedor_id'] = base.loc[base.classe_ne7 == 'DIREITO_PRODUCAO_OLEO_GAS'].apply(
    lambda r: {'AQUISICAO_PETROBRAS': 'PETROBRAS', 'AQUISICAO_MAHA': 'VENDEDORES_MAHA', 'RODADA_ANP': 'ANP'}[r.origem], axis=1)
base.loc[base.classe_ne7.isin(['ABANDONO_POCO_ARO']), 'fornecedor_id'] = 'N/A (ESTIMATIVA)'
base.loc[base.classe_ne7.isin(['BLOCOS_EXPLORATORIOS']), 'fornecedor_id'] = 'ANP'
base.loc[base.origem == 'AQUISICAO_BRAVA_JOA', 'fornecedor_id'] = '3R POTIGUAR (BRAVA)'
base['centro_custo'] = 'CC-' + base.polo.str[:3] + '-' + base.classe_ne7.str[:4]
base['status'] = base.classe_ne7.map({'IMOBILIZADO_EM_ANDAMENTO': 'EM_CONSTRUCAO', 'ALMOXARIFADO_INVERSOES_FIXAS': 'EM_ESTOQUE',
                                      'ADIANTAMENTOS': 'ADIANTAMENTO', 'BLOCOS_EXPLORATORIOS': 'EXPLORACAO'}).fillna('EM_OPERACAO')
base['data_baixa'] = pd.NaT
base['garantia_judicial'] = 'N'
i_sonda = base[(base.tipo_item.str.startswith('COMPONENTE DE SONDA'))].peso.idxmax()
base.loc[i_sonda, 'garantia_judicial'] = 'S'          # NE 7.3 - sonda dada em garantia de execucao fiscal

In [8]:
# -----------------------------------------------------------------------------
# 6) Funcoes de escala exata em CENTAVOS (metodo do maior resto)
# -----------------------------------------------------------------------------
def rateio_centavos(pesos, total_centavos):
    p = np.asarray(pesos, float); p = p / p.sum() * total_centavos
    piso = np.floor(p).astype(np.int64); falta = int(total_centavos - piso.sum())
    if falta > 0:
        piso[np.argsort(-(p - piso))[:falta]] += 1
    return piso

OFFSET = {}   # diferenca simulada de arredondamento (DF publicada em R$ mil): |offset| <= R$ 240 por grupo
def alvo_centavos(ent, classe, i):
    k = (ent, classe, i)
    if k not in OFFSET:
        OFFSET[k] = int(rng.integers(-240, 241)) if ALVO[ent][classe][i] != 0 else 0
    return (ALVO[ent][classe][i] * 1000 + OFFSET[k]) * 100

In [9]:
# -----------------------------------------------------------------------------
# 7) Duplicidades (ERRO E01) inseridas ANTES do rateio, para que o total contabil continue igual a DF
# -----------------------------------------------------------------------------
LOG = []
elig = base[base.classe_ne7.isin(['MAQUINAS_EQUIPAMENTOS', 'ALMOXARIFADO_INVERSOES_FIXAS', 'OUTROS', 'DESENVOLVIMENTO_CAMPOS']) & (base.origem != 'AQUISICAO_BRAVA_JOA')]   # UPGN preservada (totais proprios na NE 7.1)
elig = elig[elig.data_cadastro_sistema <= DATA_BASE - pd.Timedelta(days=61)]   # garante que a copia seja cadastrada DEPOIS do original
dup_src = elig.sample(160, weights=np.sqrt(elig.peso), random_state=SEED).index
dups = base.loc[dup_src].copy()
dups['usuario_cadastro'] = rng.choice(usuarios, len(dups))
dups['data_cadastro_sistema'] = (dups.data_cadastro_sistema + pd.to_timedelta(rng.integers(3, 60, len(dups)), unit='D')).clip(upper=DATA_BASE)
dups['_dup_de'] = dup_src
base['_dup_de'] = -1
base = pd.concat([base, dups], ignore_index=True)

In [10]:
# -----------------------------------------------------------------------------
# 8) Custo: rateio exato por entidade x classe (direito de producao: por polo)
# -----------------------------------------------------------------------------
base['custo_centavos'] = np.int64(0)
for ent in (ENT_PR, ENT_TI):
    for classe in ALVO[ent]:
        m = (base.entidade == ent) & (base.classe_ne7 == classe)
        if classe == 'DIREITO_PRODUCAO_OLEO_GAS':
            for polo, v in DIREITO_POR_POLO.items():
                mp = m & (base.polo == polo)
                if mp.any():
                    tot = (v * 1000 + int(rng.integers(-240, 241))) * 100
                    base.loc[mp, 'custo_centavos'] = rateio_centavos(base.loc[mp, 'peso'], tot)
        elif classe == 'MAQUINAS_EQUIPAMENTOS' and ent == ENT_PR:
            # UPGN Guamare: custo transferido na NE 7.1 (coluna "UPGN Guamare") = R$ 284.999 mil (F01 p.68-69) - alvo proprio
            mu = m & (base.origem == 'AQUISICAO_BRAVA_JOA'); tot_u = UPGN_CUSTO_MIL * 1000 * 100
            base.loc[mu, 'custo_centavos'] = rateio_centavos(base.loc[mu, 'peso'], tot_u)
            base.loc[m & ~mu, 'custo_centavos'] = rateio_centavos(base.loc[m & ~mu, 'peso'], alvo_centavos(ent, classe, 0) - tot_u)
        else:
            base.loc[m, 'custo_centavos'] = rateio_centavos(base.loc[m, 'peso'], alvo_centavos(ent, classe, 0))
# duplicatas precisam ter exatamente o mesmo valor do original: copia o valor e compensa a diferenca no item de maior valor da classe
for i in base.index[base._dup_de >= 0]:
    o = base.at[i, '_dup_de']; delta = base.at[o, 'custo_centavos'] - base.at[i, 'custo_centavos']
    base.at[i, 'custo_centavos'] = base.at[o, 'custo_centavos']
    m = (base.entidade == base.at[i, 'entidade']) & (base.classe_ne7 == base.at[i, 'classe_ne7']) & (base._dup_de < 0) & (~base.index.isin(base._dup_de)) & (base.origem != 'AQUISICAO_BRAVA_JOA')
    j = base.loc[m, 'custo_centavos'].idxmax(); base.at[j, 'custo_centavos'] -= delta

In [11]:
# -----------------------------------------------------------------------------
# 9) Depreciacao/deplecao acumulada e do exercicio
# -----------------------------------------------------------------------------
base['dep_acum_centavos'] = np.int64(0); base['dep_2025_centavos'] = np.int64(0)
base['data_inicio_depreciacao'] = pd.NaT
anos_ate_base = (DATA_BASE - base.data_aquisicao).dt.days / 365.25

# 9.1 LINEAR - calibracao dupla: (i) depreciacao acumulada = NE 7.1 e (ii) depreciacao do exercicio 2025 = coluna "Adicoes" da NE 7.1.
#     Cada item recebe uma razao r = acumulada/custo; a data de inicio da depreciacao e derivada de r (r = taxa x anos),
#     o que permite ao auditor RECALCULAR cada item. Parte dos itens mais antigos fica totalmente depreciada antes de 2025
#     (r = 1; despesa 2025 = 0) - a fracao p e achada por bissecao para que a despesa do ano bata com a NE.
#     Despesa 2025 de um item = custo x [r - max(0, r - taxa)] = custo x min(r, taxa).
CALIB_LINEAR = {}
for ent in (ENT_PR, ENT_TI):
    for classe in ('MAQUINAS_EQUIPAMENTOS', 'OUTROS'):
        m = (base.entidade == ent) & (base.classe_ne7 == classe) & (base.metodo_depreciacao == 'LINEAR')
        idx_m = base.index[m]
        c = base.loc[m, 'custo_centavos'].astype(float).values; tx = base.loc[m, 'taxa_anual_linear'].values.astype(float)
        anos = anos_ate_base[m].values.clip(min=0.02)
        fixo = (base.loc[m, 'origem'] == 'AQUISICAO_BRAVA_JOA').values
        A = float(alvo_centavos(ent, classe, 1)); D = float(alvo_centavos(ent, classe, 2))
        r = np.zeros(len(c))
        if fixo.any():                                                        # UPGN: acumulada = despesa 2025 = R$ 4.322 mil (NE 7.1)
            r[fixo] = rateio_centavos(c[fixo], UPGN_DEPREC_MIL * 1000 * 100) / c[fixo]
            A -= (c[fixo] * r[fixo]).sum(); D -= (c[fixo] * r[fixo]).sum()
        livres = np.where(~fixo)[0]
        LIM_ANOS = 25.8                                                       # inicio da operacao: fev/2000 (FRE 1.1) -> no maximo ~25,8 anos de uso
        podeF = livres[(1 / tx[livres] + 1) <= LIM_ANOS - 0.3]                # so pode estar 100% depreciado antes de 2025 quem tem vida util compativel
        ordem = podeF[np.argsort(-anos[podeF])]                               # mais antigos primeiro
        def resolver(p):
            k = int(round(p * len(ordem))); F = ordem[:k]; O = np.setdiff1d(livres, F)
            rr = np.zeros(len(c)); rr[F] = 1.0
            resto = A - c[F].sum()
            if resto <= 0 or len(O) == 0: return rr, -1.0, F, O
            nat = np.minimum(0.98, tx[O] * anos[O]); f = resto / max((c[O] * nat).sum(), 1)
            for _ in range(80):
                rO = np.minimum(0.98, nat * f); f *= resto / max((c[O] * rO).sum(), 1)
            rr[O] = np.minimum(0.98, nat * f)
            dep = (c[O] * np.minimum(rr[O], tx[O])).sum()
            return rr, dep, F, O
        lo, hi = 0.0, 1.0
        for _ in range(60):                                                   # bissecao em p (despesa decresce com p)
            p = (lo + hi) / 2; rr, dep, F, O = resolver(p)
            if dep < 0 or dep < D: hi = p
            else: lo = p
        rr, dep, F, O = resolver(lo)
        r[livres] = rr[livres]
        CALIB_LINEAR[f'{ent}|{classe}'] = dict(fracao_itens_totalmente_depreciados=round(lo, 4), despesa_alvo=D / 100, despesa_obtida=dep / 100)
        # datas coerentes com r
        anos_cont = np.where(r >= 0.999999, np.minimum(1 / tx + 1 + rng.uniform(0, 4, len(c)), LIM_ANOS), r / tx)
        base.loc[idx_m, 'data_inicio_depreciacao'] = DATA_BASE - pd.to_timedelta(np.round(anos_cont * 365.25), unit='D')
        base.loc[idx_m, 'dep_acum_centavos'] = np.round(c * r).astype(np.int64)
        base.loc[idx_m, 'dep_2025_centavos'] = np.round(c * np.where(r >= 0.999999, 0.0, np.minimum(r, tx))).astype(np.int64)
        base.loc[idx_m[fixo], 'dep_2025_centavos'] = base.loc[idx_m[fixo], 'dep_acum_centavos']

# 9.2 UNIDADES PRODUZIDAS: acumulada por idade (escala exata) e deplecao 2025 = VCL x t/(1-t), t = taxa do campo x k calibrado
tx_campo = campos.set_index('campo')
for ent in (ENT_PR, ENT_TI):
    for classe, col_tx in (('DIREITO_PRODUCAO_OLEO_GAS', 'taxa_uop_1P_2025'), ('DESENVOLVIMENTO_CAMPOS', 'taxa_uop_PDP_2025'), ('ABANDONO_POCO_ARO', 'taxa_uop_PDP_2025')):
        m = (base.entidade == ent) & (base.classe_ne7 == classe)
        if not m.any(): continue
        c = base.loc[m, 'custo_centavos'].astype(float).values
        anos = anos_ate_base[m].values.clip(min=0)
        r0 = (1 - np.exp(-0.16 * anos)) * rng.uniform(0.7, 1.3, m.sum())
        alvo_acum = alvo_centavos(ent, classe, 1); f = alvo_acum / max((c * r0).sum(), 1)
        for _ in range(60):
            r = np.minimum(0.985, r0 * f); f *= alvo_acum / max((c * r).sum(), 1)
        r = np.minimum(0.985, r0 * f)
        base.loc[m, 'dep_acum_centavos'] = rateio_centavos(np.maximum(c * r, 1e-9), alvo_acum) if alvo_acum > 0 else 0
        vcl = c - base.loc[m, 'dep_acum_centavos'].values
        t = base.loc[m, 'campo'].map(tx_campo[col_tx]).values
        frac = np.where(base.loc[m, 'data_aquisicao'].dt.year.values == 2025, (DATA_BASE - base.loc[m, 'data_aquisicao']).dt.days.values / 365.0, 1.0)
        alvo_dep = alvo_centavos(ent, classe, 2)
        lo, hi = 1e-6, 1 / max(t.max(), 1e-9) * 0.95
        for _ in range(100):                                                  # bissecao do fator k
            k = (lo + hi) / 2; tk = np.minimum(t * k, 0.9)
            tot = (vcl * tk / (1 - tk) * frac).sum()
            lo, hi = (k, hi) if tot < alvo_dep else (lo, k)
        tk = np.minimum(t * k, 0.9)
        base.loc[m, 'dep_2025_centavos'] = rateio_centavos(np.maximum(vcl * tk / (1 - tk) * frac, 1e-9), alvo_dep)
        base.loc[m, 'fator_calibracao_uop'] = k
        base.loc[m, 'data_inicio_depreciacao'] = base.loc[m, 'data_aquisicao']
base['taxa_uop_campo_2025'] = np.where(base.metodo_depreciacao == 'UOP_1P', base.campo.map(tx_campo.taxa_uop_1P_2025),
                              np.where(base.metodo_depreciacao == 'UOP_PDP', base.campo.map(tx_campo.taxa_uop_PDP_2025), np.nan))
# duplicatas: mesma acumulada do original (o erro e a duplicidade integral do registro)
for i in base.index[base._dup_de >= 0]:
    o = base.at[i, '_dup_de']
    for col in ('dep_acum_centavos', 'dep_2025_centavos', 'data_inicio_depreciacao'):
        base.at[i, col] = base.at[o, col]

def log(tipo, idx, descricao, efeito_centavos, campo_alterado='', original='', inserido=''):
    for k, i in enumerate(idx):
        LOG.append(dict(tipo_erro=tipo, id_interno=int(i), descricao=descricao, efeito_monetario_centavos=int(efeito_centavos[k]) if np.ndim(efeito_centavos) else int(efeito_centavos),
                        campo_alterado=campo_alterado, valor_original=str(original[k]) if np.ndim(original) else str(original),
                        valor_inserido=str(inserido[k]) if np.ndim(inserido) else str(inserido)))

vcl_tmp = base.custo_centavos - base.dep_acum_centavos
log('E01', base.index[base._dup_de >= 0], 'Registro duplicado (mesmo documento, fornecedor, valor e data de aquisicao do registro original)',
    vcl_tmp[base._dup_de >= 0].values, 'registro inteiro', base.loc[base._dup_de >= 0, '_dup_de'].values, 'copia')

In [12]:
# -----------------------------------------------------------------------------
# 10) INSERCAO DOS DEMAIS ERROS (registro no LOG; os testes NAO leem o LOG)
# -----------------------------------------------------------------------------
usado = set(base.index[base._dup_de >= 0]) | set(base._dup_de[base._dup_de >= 0]) | {i_sonda}
def escolher(mask, n, peso=None, orcamento=None, valor=None):
    """sorteio sem reposicao (ponderado pelo valor quando 'peso' e informado - metodo de Efraimidis-Spirakis);
    com 'orcamento', para quando o efeito monetario acumulado atinge o orcamento (em centavos)."""
    cand = base.index[mask & ~base.index.isin(list(usado)) & (base.origem != 'AQUISICAO_BRAVA_JOA')]   # itens da UPGN nao recebem erros
    if peso is None:
        ordem = rng.permutation(cand)
    else:
        w = np.maximum(base.loc[cand, peso].astype(float).values, 1.0)
        ordem = cand[np.argsort(-np.log(rng.random(len(cand))) / w)]
    ordem = np.asarray(ordem)
    if orcamento is not None:
        acum = np.cumsum(valor.loc[ordem[:n]].values)
        ordem = ordem[:max(1, np.searchsorted(acum, orcamento) + 1)]
    ordem = ordem[:n]
    usado.update(ordem.tolist()); return ordem

VCL = lambda: base.custo_centavos - base.dep_acum_centavos
nao_dep = base.metodo_depreciacao == 'NAO_DEPRECIA'
linear = base.metodo_depreciacao == 'LINEAR'

# E02 - corte: item capitalizado em 2026 (posterior a data-base) incluido no saldo de 31/12/2025
faixa = lambda a, b: (VCL() >= a * 100) & (VCL() <= b * 100)          # faixa de valor liquido (R$) dos itens elegiveis
base['_raiz'] = np.sqrt(base.custo_centavos.astype(float))
ix = escolher(base.classe_ne7.isin(['MAQUINAS_EQUIPAMENTOS', 'ALMOXARIFADO_INVERSOES_FIXAS', 'IMOBILIZADO_EM_ANDAMENTO', 'OUTROS', 'DESENVOLVIMENTO_CAMPOS'])
              & faixa(5_000, 400_000), 90, orcamento=250_000_000, valor=VCL(), peso='_raiz')
novas = pd.Timestamp('2026-01-02') + pd.to_timedelta(rng.integers(0, 75, len(ix)), unit='D')
log('E02', ix, 'Corte: data de aquisicao/capitalizacao posterior a 31/12/2025 (item nao existia na data-base)', VCL().loc[ix].values, 'data_aquisicao',
    base.loc[ix, 'data_aquisicao'].dt.date.values, novas.date)
base.loc[ix, 'data_aquisicao'] = novas; base.loc[ix, 'data_cadastro_sistema'] = novas; base.loc[ix, 'data_ultima_alteracao'] = novas
base.loc[ix, 'data_inicio_depreciacao'] = novas
# o saldo contabil (custo) e mantido; acumulada mantida (o erro e de existencia/corte, nao de calculo)

# E03 - depreciacao acumulada maior que o custo (valor liquido negativo) - itens lineares
ix = escolher(linear & (base.custo_centavos > 2_000_000), 40)
nova = (base.loc[ix, 'custo_centavos'] * rng.uniform(1.05, 1.6, len(ix))).round().astype(np.int64)
log('E03', ix, 'Depreciacao acumulada superior ao custo (valor contabil liquido negativo)', -(nova - base.loc[ix, 'custo_centavos']).values,
    'dep_acum_centavos', base.loc[ix, 'dep_acum_centavos'].values, nova.values)
base.loc[ix, 'dep_acum_centavos'] = nova.values

# E04 - depreciacao acumulada subavaliada (diverge do recalculo pela taxa e pela data de inicio)
ix = escolher(linear & (base.dep_acum_centavos > 500_000) & (base.origem != 'AQUISICAO_BRAVA_JOA'), 320, orcamento=150_000_000,
              valor=(base.dep_acum_centavos * 0.45), peso='dep_acum_centavos')
fator = rng.uniform(0.25, 0.65, len(ix)); nova = (base.loc[ix, 'dep_acum_centavos'] * (1 - fator)).round().astype(np.int64)
log('E04', ix, 'Depreciacao acumulada registrada a menor que o recalculo (taxa x tempo de uso)', (base.loc[ix, 'dep_acum_centavos'] - nova).values,
    'dep_acum_centavos', base.loc[ix, 'dep_acum_centavos'].values, nova.values)
base.loc[ix, 'dep_acum_centavos'] = nova.values

# E05 - item baixado (sucateado/alienado em 2025) ainda com saldo
ix = escolher(base.classe_ne7.isin(['MAQUINAS_EQUIPAMENTOS', 'OUTROS', 'DESENVOLVIMENTO_CAMPOS']) & faixa(3_000, 300_000), 130, orcamento=250_000_000, valor=VCL(), peso='_raiz')
db = pd.Timestamp('2025-01-15') + pd.to_timedelta(rng.integers(0, 340, len(ix)), unit='D')
log('E05', ix, 'Item com status BAIXADO (baixa em 2025) mantido com saldo no subrazao', VCL().loc[ix].values, 'status/data_baixa', 'EM_OPERACAO', 'BAIXADO')
base.loc[ix, 'status'] = 'BAIXADO'; base.loc[ix, 'data_baixa'] = db.values

# E06 - almoxarifado sem movimentacao ha mais de 5 anos (risco de obsolescencia/valor recuperavel)
ix = escolher(base.classe_ne7 == 'ALMOXARIFADO_INVERSOES_FIXAS', 95)
novas = datas_unif('2014-01-01', '2020-10-31', len(ix))
log('E06', ix, 'Material de almoxarifado de investimento sem movimentacao ha mais de 5 anos (indicio de obsolescencia)', 0, 'data_aquisicao',
    base.loc[ix, 'data_aquisicao'].dt.date.values, novas.date)
base.loc[ix, 'data_aquisicao'] = novas.values; base.loc[ix, 'data_cadastro_sistema'] = novas.values; base.loc[ix, 'data_ultima_alteracao'] = novas.values

# E07 - falha de segregacao (maker-checker): aprovacao ausente ou feita pelo proprio cadastrador
ix = escolher(pd.Series(True, index=base.index), 620)
metade = len(ix) // 2
log('E07', ix[:metade], 'Cadastro sem aprovacao (campo usuario_aprovacao vazio)', 0, 'usuario_aprovacao', base.loc[ix[:metade], 'usuario_aprovacao'].values, '')
log('E07', ix[metade:], 'Cadastro aprovado pelo proprio usuario que cadastrou (maker = checker)', 0, 'usuario_aprovacao', base.loc[ix[metade:], 'usuario_aprovacao'].values,
    base.loc[ix[metade:], 'usuario_cadastro'].values)
base.loc[ix[:metade], 'usuario_aprovacao'] = ''
base.loc[ix[metade:], 'usuario_aprovacao'] = base.loc[ix[metade:], 'usuario_cadastro'].values

# E08 - item vinculado a campo inexistente no cadastro de concessoes e sem documento de suporte
ix = escolher(base.classe_ne7.isin(['DESENVOLVIMENTO_CAMPOS', 'MAQUINAS_EQUIPAMENTOS']) & faixa(3_000, 150_000), 55, orcamento=120_000_000, valor=VCL(), peso='_raiz')
falsos = rng.choice(['CAMPO X-01', 'FAZENDA NOVA', 'POT-CONC-99 (SINTETICO)'], len(ix))
log('E08', ix, 'Campo nao cadastrado entre as 58 concessoes e documento de suporte inexistente (existencia)', VCL().loc[ix].values, 'campo', base.loc[ix, 'campo'].values, falsos)
base.loc[ix, 'campo'] = falsos

# E09 - superavaliacao de custo (despesa capitalizada / valor acima do documento) - so aparece na inspecao documental
VCLs = VCL()
ix = escolher(base.classe_ne7.isin(['DESENVOLVIMENTO_CAMPOS', 'MAQUINAS_EQUIPAMENTOS', 'ALMOXARIFADO_INVERSOES_FIXAS', 'IMOBILIZADO_EM_ANDAMENTO'])
              & (VCLs >= 2_000_000) & (VCLs <= 300_000_000), 400, peso='_raiz', orcamento=800_000_000, valor=VCLs * 0.4)
taint = np.where(rng.random(len(ix)) < 0.06, 1.0, rng.uniform(0.05, 0.75, len(ix)))
efeito = np.round(VCLs.loc[ix].values * taint).astype(np.int64)
log('E09', ix, 'Custo registrado acima do suportado por documento (inclui despesa de manutencao capitalizada)', efeito, 'valor suportado (arquivo de inspecao)',
    VCLs.loc[ix].values, (VCLs.loc[ix].values - efeito))
E09_SUPORTE = dict(zip(ix, VCLs.loc[ix].values - efeito))

# E10 - imobilizado em andamento parado ha mais de 2 anos (indicio de impairment / transferencia tardia)
ix = escolher(base.classe_ne7 == 'IMOBILIZADO_EM_ANDAMENTO', 45)
novas = datas_unif('2019-01-01', '2023-09-30', len(ix))
log('E10', ix, 'Ordem em andamento sem movimentacao ha mais de 2 anos', 0, 'data_aquisicao', base.loc[ix, 'data_aquisicao'].dt.date.values, novas.date)
base.loc[ix, 'data_aquisicao'] = novas.values; base.loc[ix, 'data_cadastro_sistema'] = novas.values; base.loc[ix, 'data_ultima_alteracao'] = novas.values

# E11 - data de aquisicao anterior a constituicao da companhia (21/07/1999) em item de capex proprio
ix = escolher((base.origem == 'CAPEX_PROPRIO') & (base.classe_ne7 == 'OUTROS') & linear, 30)
novas = datas_unif('1985-01-01', '1999-06-30', len(ix))
log('E11', ix, 'Data de aquisicao anterior a constituicao da companhia (21/07/1999) - erro cadastral', 0, 'data_aquisicao', base.loc[ix, 'data_aquisicao'].dt.date.values, novas.date)
base.loc[ix, 'data_aquisicao'] = novas.values

In [13]:
# -----------------------------------------------------------------------------
# 11) Restabelece o total EXATO da acumulada por entidade x classe (compensa E03/E04 em itens sem erro)
#     -> o total contabil da base continua igual ao saldo publicado (a DF "contem" as distorcoes simuladas)
# -----------------------------------------------------------------------------
com_erro = set(r['id_interno'] for r in LOG) | set(base._dup_de[base._dup_de >= 0])
for ent in (ENT_PR, ENT_TI):
    for classe in ALVO[ent]:
        m = (base.entidade == ent) & (base.classe_ne7 == classe)
        alvo = alvo_centavos(ent, classe, 1)
        dif = alvo - base.loc[m, 'dep_acum_centavos'].sum()
        if dif == 0: continue
        livre = m & ~base.index.isin(list(com_erro)) & (base.origem != 'AQUISICAO_BRAVA_JOA') & (base.dep_acum_centavos < 0.97 * base.custo_centavos)
        folga = (base.custo_centavos - base.dep_acum_centavos) if dif > 0 else base.dep_acum_centavos   # quanto cada item comporta
        if classe in ('MAQUINAS_EQUIPAMENTOS', 'OUTROS'):                     # lineares: so itens com r >= taxa, para nao alterar a despesa 2025 (= custo x taxa)
            tx_ = base.taxa_anual_linear.fillna(0)
            folga = ((0.97 * base.custo_centavos - base.dep_acum_centavos) if dif > 0 else (base.dep_acum_centavos - tx_ * base.custo_centavos)).clip(lower=0).apply(np.floor)
            livre = livre & (base.dep_acum_centavos >= tx_ * base.custo_centavos)
        cand = folga[livre & (folga > 0)].nlargest(5000).index
        aj = rateio_centavos(folga.loc[cand].astype(float), abs(dif)) * np.sign(dif)
        base.loc[cand, 'dep_acum_centavos'] += aj
        mm = cand[base.loc[cand, 'metodo_depreciacao'] == 'LINEAR']
        if len(mm):                                                             # mantem coerencia data de inicio x acumulada
            r = base.loc[mm, 'dep_acum_centavos'] / base.loc[mm, 'custo_centavos']
            base.loc[mm, 'data_inicio_depreciacao'] = DATA_BASE - pd.to_timedelta(np.round(r / base.loc[mm, 'taxa_anual_linear'] * 365.25), unit='D')
            base.loc[mm, 'dep_2025_centavos'] = np.round(base.loc[mm, 'custo_centavos'] * np.minimum(r, base.loc[mm, 'taxa_anual_linear'])).astype(np.int64)

# coerencia cadastral: nos itens lineares sem erro inserido, a data de aquisicao nao pode ser posterior ao inicio da depreciacao
lin_ok = (base.metodo_depreciacao == 'LINEAR') & ~base.index.isin(list(com_erro)) & (base.data_inicio_depreciacao < base.data_aquisicao)
base.loc[lin_ok, 'data_aquisicao'] = base.loc[lin_ok, 'data_inicio_depreciacao']
base.loc[lin_ok, 'data_cadastro_sistema'] = base.loc[lin_ok, ['data_cadastro_sistema', 'data_aquisicao']].min(axis=1)
base.loc[lin_ok, 'data_cadastro_sistema'] = (base.loc[lin_ok, 'data_aquisicao'] + pd.to_timedelta(rng.integers(0, 25, lin_ok.sum()), unit='D')).clip(upper=DATA_BASE)
base.loc[lin_ok, 'data_ultima_alteracao'] = base.loc[lin_ok, ['data_ultima_alteracao', 'data_cadastro_sistema']].max(axis=1)

In [14]:
# -----------------------------------------------------------------------------
# 12) Montagem final, identificadores e arquivos
# -----------------------------------------------------------------------------
base = base.sample(frac=1, random_state=SEED).reset_index().rename(columns={'index': 'id_interno'})   # embaralha (ordem nao revela classe)
base['id_registro'] = 'IMB-' + (base.index + 1).astype(str).str.zfill(7)
mapa_id = dict(zip(base.id_interno, base.id_registro))
base['custo_historico'] = base.custo_centavos / 100
base['depreciacao_acumulada'] = base.dep_acum_centavos / 100
base['valor_contabil_liquido'] = (base.custo_centavos - base.dep_acum_centavos) / 100
base['depreciacao_2025'] = base.dep_2025_centavos / 100
base = base.merge(plano[['conta_custo', 'descricao_conta', 'conta_depreciacao']], on='conta_custo', how='left')
base['descricao_conta_depreciacao'] = base.conta_depreciacao.map(DESC_DEP).fillna('')
base['natureza_dado'] = 'SINTETICO'
COLS = ['id_registro', 'entidade', 'polo', 'campo', 'uf', 'poco_id', 'classe_ne7', 'conta_custo', 'descricao_conta', 'conta_depreciacao',
        'descricao_conta_depreciacao', 'tipo_item', 'origem', 'data_aquisicao', 'data_inicio_depreciacao', 'metodo_depreciacao', 'taxa_anual_linear',
        'taxa_uop_campo_2025', 'custo_historico', 'depreciacao_acumulada', 'valor_contabil_liquido', 'depreciacao_2025', 'status', 'data_baixa',
        'documento_origem', 'fornecedor_id', 'centro_custo', 'garantia_judicial', 'usuario_cadastro', 'usuario_aprovacao', 'data_cadastro_sistema',
        'data_ultima_alteracao', 'natureza_dado']
saida = base[COLS].copy()
for c in ('data_aquisicao', 'data_inicio_depreciacao', 'data_baixa', 'data_cadastro_sistema', 'data_ultima_alteracao'):
    saida[c] = pd.to_datetime(saida[c]).dt.strftime('%Y-%m-%d')
saida.to_csv(f'{OUT}/BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz', index=False, sep=';', decimal=',', encoding='utf-8', compression={'method': 'gzip', 'mtime': 0})
saida.head(2000).to_csv(f'{OUT}/BASE_IMOBILIZADO_RECV3_SINTETICA_amostra_2000_linhas.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

# Arquivo SIMULADO de resultado da inspecao documental (o "documento" que o auditor veria ao inspecionar o item)
erro_por_id = {}
for r in LOG:
    erro_por_id.setdefault(r['id_interno'], []).append(r['tipo_erro'])
sup = base[['id_interno', 'id_registro', 'documento_origem', 'fornecedor_id', 'data_aquisicao', 'custo_centavos', 'dep_acum_centavos']].copy()
sup['documento_localizado'] = 'S'; sup['valor_liquido_suportado'] = (sup.custo_centavos - sup.dep_acum_centavos) / 100
sup['data_documento'] = pd.to_datetime(sup.data_aquisicao).dt.strftime('%Y-%m-%d'); sup['observacao_inspecao'] = 'Documento confere com o registro'
dep_ok = {}
for r in LOG:
    if r['tipo_erro'] in ('E04',): dep_ok[r['id_interno']] = int(r['valor_original'])
idx_err = sup.index[sup.id_interno.isin(list(erro_por_id.keys())) | (base._dup_de >= 0)]
for k in idx_err:
    row = sup.loc[k]; tipos = erro_por_id.get(row.id_interno, [])
    if base.at[k, '_dup_de'] >= 0:
        sup.at[k, 'valor_liquido_suportado'] = 0.0; sup.at[k, 'observacao_inspecao'] = f'Documento ja vinculado ao registro {mapa_id[base.at[k, "_dup_de"]]} (duplicidade)'
    if 'E02' in tipos:
        sup.at[k, 'valor_liquido_suportado'] = 0.0; sup.at[k, 'observacao_inspecao'] = 'Documento datado de 2026: bem recebido apos a data-base'
    if 'E05' in tipos:
        sup.at[k, 'valor_liquido_suportado'] = 0.0; sup.at[k, 'observacao_inspecao'] = 'Laudo de sucateamento/baixa emitido em 2025'
    if 'E08' in tipos:
        sup.at[k, 'documento_localizado'] = 'N'; sup.at[k, 'valor_liquido_suportado'] = 0.0; sup.at[k, 'observacao_inspecao'] = 'Documento nao localizado; campo inexistente'
    if 'E09' in tipos:
        sup.at[k, 'valor_liquido_suportado'] = E09_SUPORTE[row.id_interno] / 100; sup.at[k, 'observacao_inspecao'] = 'Nota/medicao suporta valor inferior ao registrado'
    if 'E04' in tipos:
        sup.at[k, 'valor_liquido_suportado'] = (row.custo_centavos - dep_ok[row.id_interno]) / 100; sup.at[k, 'observacao_inspecao'] = 'Recalculo da depreciacao diverge do registro'
    if 'E03' in tipos:
        sup.at[k, 'valor_liquido_suportado'] = 0.0; sup.at[k, 'observacao_inspecao'] = 'Bem totalmente depreciado; acumulada excede o custo'
sup['natureza_dado'] = 'SINTETICO - resultado simulado de inspecao documental (usado apenas para itens selecionados)'
sup[['id_registro', 'documento_origem', 'documento_localizado', 'data_documento', 'valor_liquido_suportado', 'observacao_inspecao', 'natureza_dado']].to_csv(
    f'{OUT}/INSPECAO_DOCUMENTAL_SIMULADA.csv.gz', index=False, sep=';', decimal=',', encoding='utf-8', compression={'method': 'gzip', 'mtime': 0})

# Parametros e cadastros auxiliares
campos.drop(columns=['sigla']).to_csv(f'{OUT}/PARAMETROS_CAMPOS_SIMULADOS.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
plano.to_csv(f'{OUT}/PLANO_CONTAS_IMOBILIZADO_SIMULADO.csv', index=False, sep=';', encoding='utf-8-sig')
pocos[['poco_id', 'campo', 'polo', 'entidade', 'uf', 'data_entrada', 'origem_poco']].assign(natureza_dado='SINTETICO').to_csv(
    f'{OUT}/CADASTRO_POCOS_SIMULADO.csv', index=False, sep=';', encoding='utf-8-sig')

# LOG dos erros inseridos (controle do preparador; os testes so o leem na etapa final de comparacao)
lg = pd.DataFrame(LOG); lg['id_registro'] = lg.id_interno.map(mapa_id); lg['efeito_monetario_R$'] = lg.efeito_monetario_centavos / 100
lg = lg.drop(columns=['id_interno', 'efeito_monetario_centavos']).sort_values(['tipo_erro', 'id_registro'])
lg.insert(0, 'seq', range(1, len(lg) + 1))
lg.to_csv(f'{CTRL}/LOG_ERROS_INSERIDOS.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

meta = dict(semente=SEED, linhas=int(len(saida)), data_base='2025-12-31', gerado_por='codigo/gerar_base_imobilizado.py',
            sha256_conteudo_csv=hashlib.sha256(__import__('gzip').open(f'{OUT}/BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz', 'rb').read()).hexdigest(),   # hash do CSV descompactado (o cabecalho gzip guarda a hora)
            total_custo=float(saida.custo_historico.sum()), total_dep_acum=float(saida.depreciacao_acumulada.sum()),
            total_liquido=float(saida.valor_contabil_liquido.sum()), erros_inseridos=lg.tipo_erro.value_counts().sort_index().to_dict(),
            calibracao_depreciacao_linear=CALIB_LINEAR, offsets_arredondamento_reais={f'{k[0]}|{k[1]}|{["custo","acum","dep2025"][k[2]]}': v for k, v in OFFSET.items()})
json.dump(meta, open(f'{OUT}/METADADOS_GERACAO.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(json.dumps({k: meta[k] for k in ('linhas', 'total_custo', 'total_dep_acum', 'total_liquido', 'erros_inseridos')}, ensure_ascii=False))

{"linhas": 532598, "total_custo": 9703550773.999998, "total_dep_acum": 3532711454.9999995, "total_liquido": 6170839319.000002, "erros_inseridos": {"E01": 160, "E02": 46, "E03": 40, "E04": 70, "E05": 63, "E06": 95, "E07": 620, "E08": 35, "E09": 97, "E10": 45, "E11": 30}}


## C. Scripts de auditoria: conciliação, testes na população, MUS e comparação
> Procedimentos **simulados para fins didáticos** (executados sobre dados sintéticos).

Código-fonte: `codigo/testes_auditoria_base.py` (o mesmo código, dividido em células).

In [15]:
# -*- coding: utf-8 -*-
# =============================================================================
# P2 - AUDITORIA 1 (UnB 2026.2) - PetroReconcavo S.A. (RECV3) - exercicio 2025
# Preparador: matricula 211059370
#
# >>> SECAO DE SCRIPTS DE AUDITORIA (testes sobre a BASE SIMULADA) <<<
# Procedimentos classificados como "SIMULADO PARA FINS DIDATICOS": executados
# sobre DADOS SINTETICOS. Nenhum resultado aqui e constatacao sobre a companhia real.
#
# Ordem:  (1) conciliacao base x NE 7 / CVM
#         (2) testes na populacao inteira (T01-T10) - feitos "as cegas" (sem ler o log de erros)
#         (3) amostragem MUS (Gramling, cap. 8; NBC TA 530) - selecao e avaliacao (LSI)
#         (4) SO ENTAO le o log de erros inseridos e compara inseridos x detectados
# Uso:  python codigo/testes_auditoria_base.py     (na raiz P2_PetroReconcavo; apos gerar a base)
# =============================================================================
import os, json
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

DATA_BASE = pd.Timestamp('2025-12-31')
B = 'base_simulada'; RES = 'base_simulada/resultados'; os.makedirs(RES, exist_ok=True); os.makedirs('planilhas', exist_ok=True)
SEED_MUS = 211059370 + 530          # semente do ponto de partida aleatorio do MUS (reprodutivel)

In [16]:
# ---------------------------------------------------------------- parametros do auditor - definidos no PT 320 (NBC TA 320 (R1))
# MG  (item 10; A4-A8): referencial = RECEITA LIQUIDA consolidada 2025 (R$ 3.157.609 mil, F01 DRE p.52), porque o lucro antes
#      do imposto e VOLATIL (2024: R$ 363.405 mil; 2025: R$ 776.194 mil, com R$ 366.796 mil de swaps) - item A5 ("quando o lucro
#      antes do imposto ... e volatil, outros referenciais podem ser mais apropriados, como lucro bruto ou receita total").
#      Percentual: 1% (exemplo do item A8 para receita total) -> R$ 31.576 mil, arredondado para baixo: R$ 31.500 mil.
# ME  (item 11; A13): 65% da MG - julgamento (a norma nao fixa percentual): primeira auditoria (sem historico de distorcoes,
#      A13), inconsistencias de divulgacao ja identificadas e risco significativo de impairment -> expectativa de distorcoes.
# IT  da populacao do imobilizado = ME (nao foi fixada materialidade especifica para o imobilizado - item 10/A11).
# RA, RI, RC, IE: modelo de risco do material de MUS (Gramling) e PT 315/330.
PARAM = dict(
    MG=31_500_000.00,          # materialidade para as DCs como um todo (R$) - PT 320
    pct_ME=0.65,               # materialidade para execucao = 65% da MG (PT 320, A13)
    RA=0.05,                   # risco de auditoria aceitavel
    RI=1.00,                   # risco inerente do imobilizado: maximo (estimativas UoP, impairment, capitalizacao - PT 315)
    RC=1.00,                   # risco de controle: maximo (sem teste de controles - sem acesso a entidade)
    pct_IE=0.10,               # incorrecao esperada = 10% da incorrecao toleravel (sem historico - primeira auditoria)
)
PARAM['ME'] = PARAM['MG'] * PARAM['pct_ME']; PARAM['IT'] = PARAM['ME']; PARAM['IE'] = PARAM['IT'] * PARAM['pct_IE']

# Ilustracao 8.7 (Gramling) - fatores de confiabilidade, expansao e ajuste incremental (5 paginas do material da disciplina)
TAB87 = pd.DataFrame({
    'RD': [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50],
    'FC': [4.61, 3.00, 2.31, 1.90, 1.61, 1.39, 1.21, 0.70],
    'FEE': [1.90, 1.60, 1.50, 1.40, 1.30, 1.25, 1.20, 1.00]})
INCR = {0.01: [1.03, .77, .64, .56, .50, .46, .43, .41, .38, .38], 0.05: [.75, .55, .46, .40, .36, .33, .30, .29, .27, .26],
        0.10: [.58, .44, .36, .31, .28, .26, .24, .22, .21, .20], 0.15: [.48, .34, .30, .25, .23, .21, .19, .18, .17, .17],
        0.20: [.39, .28, .24, .21, .18, .17, .16, .14, .14, .14], 0.25: [.31, .23, .18, .17, .15, .13, .13, .12, .11, .10],
        0.30: [.23, .18, .15, .13, .11, .11, .10, .09, .08, .08], 0.50: [0] * 10}

def ler_base():
    return pd.read_csv(f'{B}/BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz', sep=';', decimal=',', low_memory=False,
                       parse_dates=['data_aquisicao', 'data_inicio_depreciacao', 'data_baixa', 'data_cadastro_sistema', 'data_ultima_alteracao'],
                       dtype={'usuario_aprovacao': str, 'poco_id': str})

base = ler_base()
base['usuario_aprovacao'] = base.usuario_aprovacao.fillna('')
campos = pd.read_csv(f'{B}/PARAMETROS_CAMPOS_SIMULADOS.csv', sep=';', decimal=',', encoding='utf-8-sig')
print('linhas lidas:', len(base))

linhas lidas: 532598


In [17]:
# =============================================================================
# (1) CONCILIACAO - base x NE 7.1 (F01 p.68-69) x CVM (F03, conta 1.02.03)
# =============================================================================
NE7_CTR = {'MAQUINAS_EQUIPAMENTOS': (716487, 103733), 'IMOBILIZADO_EM_ANDAMENTO': (68419, 0), 'DIREITO_PRODUCAO_OLEO_GAS': (2851955, 850980),
           'DESENVOLVIMENTO_CAMPOS': (4018461, 1749489), 'BLOCOS_EXPLORATORIOS': (9553, 0), 'ABANDONO_POCO_ARO': (73827, 36676),
           'ALMOXARIFADO_INVERSOES_FIXAS': (435242, 0), 'ADIANTAMENTOS': (15677, 0), 'OUTROS': (195997, 39032)}
NE7_CON = {'MAQUINAS_EQUIPAMENTOS': (718521, 103788, 38884), 'IMOBILIZADO_EM_ANDAMENTO': (72171, 0, 0), 'DIREITO_PRODUCAO_OLEO_GAS': (2931329, 922414, 132263),
           'DESENVOLVIMENTO_CAMPOS': (5186676, 2422704, 485386), 'BLOCOS_EXPLORATORIOS': (20157, 0, 0), 'ABANDONO_POCO_ARO': (79303, 41848, 1174),
           'ALMOXARIFADO_INVERSOES_FIXAS': (477651, 0, 0), 'ADIANTAMENTOS': (18419, 0, 0), 'OUTROS': (199325, 41957, 14161)}
g = base.groupby(['entidade', 'classe_ne7'])[['custo_historico', 'depreciacao_acumulada', 'valor_contabil_liquido', 'depreciacao_2025']].sum()
conc = []
for classe, (c, d, dep) in NE7_CON.items():
    pr = g.loc[('PETRORECONCAVO', classe)] if ('PETRORECONCAVO', classe) in g.index else pd.Series(0, index=g.columns)
    ti = g.loc[('SPE_TIETA', classe)] if ('SPE_TIETA', classe) in g.index else pd.Series(0, index=g.columns)
    conc.append(dict(classe_ne7=classe, n_registros=int((base.classe_ne7 == classe).sum()),
                     custo_base_ctr=pr.custo_historico, custo_ne7_ctr=NE7_CTR[classe][0] * 1000.0, custo_base_cons=pr.custo_historico + ti.custo_historico, custo_ne7_cons=c * 1000.0,
                     acum_base_ctr=pr.depreciacao_acumulada, acum_ne7_ctr=NE7_CTR[classe][1] * 1000.0, acum_base_cons=pr.depreciacao_acumulada + ti.depreciacao_acumulada, acum_ne7_cons=d * 1000.0,
                     dep2025_base_cons=pr.depreciacao_2025 + ti.depreciacao_2025, dep2025_ne7_cons=dep * 1000.0))
conc = pd.DataFrame(conc)
balancete = base.groupby(['conta_custo', 'descricao_conta'])[['custo_historico', 'depreciacao_acumulada', 'valor_contabil_liquido']].sum().reset_index()
balancete['n_registros'] = base.groupby('conta_custo').size().values

In [18]:
# =============================================================================
# (2) TESTES NA POPULACAO INTEIRA (100% dos registros - nao e amostragem, NBC TA 500 A52-A56 / 530 item 5)
# =============================================================================
exc = []
def registrar(teste, df, detalhe, valor=None):
    v = df.valor_contabil_liquido if valor is None else valor
    exc.append(pd.DataFrame({'teste': teste, 'id_registro': df.id_registro.values, 'classe_ne7': df.classe_ne7.values,
                             'valor_contabil_liquido': df.valor_contabil_liquido.values, 'valor_em_risco': np.asarray(v, float), 'detalhe': detalhe}))

# T01 duplicidade: mesma chave (documento, fornecedor, custo e data de aquisicao); excecao = registro cadastrado depois
chave = ['documento_origem', 'fornecedor_id', 'custo_historico', 'data_aquisicao']
d = base[base.duplicated(chave, keep=False)].sort_values(chave + ['data_cadastro_sistema', 'id_registro'])
d1 = d[d.duplicated(chave, keep='first')]
registrar('T01_DUPLICIDADE', d1, 'Mesmo documento/fornecedor/valor/data de outro registro (cadastro posterior)')
# T02 corte: data de aquisicao posterior a data-base
registrar('T02_CORTE', base[base.data_aquisicao > DATA_BASE], 'Data de aquisicao posterior a 31/12/2025')
# T03 valor liquido negativo
t3 = base[base.valor_contabil_liquido < 0]; registrar('T03_VCL_NEGATIVO', t3, 'Depreciacao acumulada maior que o custo', valor=t3.valor_contabil_liquido)
# T04 recalculo da depreciacao linear (taxa x tempo desde o inicio da depreciacao; tolerancia = maior entre 1% do custo e R$ 100)
L = base[base.metodo_depreciacao == 'LINEAR']
anos = ((DATA_BASE - L.data_inicio_depreciacao).dt.days / 365.25).clip(lower=0)
esperada = (L.custo_historico * np.minimum(1, anos * L.taxa_anual_linear)).round(2)
dif = L.depreciacao_acumulada - esperada
f4 = abs(dif) > np.maximum(0.01 * L.custo_historico, 100)
registrar('T04_RECALCULO_DEPRECIACAO', L[f4], 'Acumulada difere do recalculo (taxa x tempo)', valor=-dif[f4])
# T05 item baixado ainda com saldo
registrar('T05_BAIXADO_COM_SALDO', base[(base.status == 'BAIXADO') & (base.valor_contabil_liquido != 0)], 'Status BAIXADO com saldo remanescente')
# T06 almoxarifado sem movimentacao ha mais de 5 anos
registrar('T06_ALMOX_SEM_MOVIMENTO_5A', base[(base.classe_ne7 == 'ALMOXARIFADO_INVERSOES_FIXAS') & (base.data_ultima_alteracao < pd.Timestamp('2021-01-01'))],
          'Sem movimentacao desde antes de 01/01/2021 (avaliar obsolescencia / valor recuperavel)')
# T07 segregacao de funcoes no cadastro (maker-checker)
registrar('T07_MAKER_CHECKER', base[(base.usuario_aprovacao == '') | (base.usuario_aprovacao == base.usuario_cadastro)],
          'Cadastro sem aprovacao ou aprovado pelo proprio cadastrador', valor=0)
# T08 campo inexistente no cadastro de concessoes (existencia / classificacao)
validos = set(campos.campo) | {'UPGN GUAMARE'} | set(base.loc[base.classe_ne7 == 'BLOCOS_EXPLORATORIOS', 'campo'])
registrar('T08_CAMPO_NAO_CADASTRADO', base[~base.campo.isin(validos)], 'Campo/local nao consta das 58 concessoes, blocos ou UPGN')
# T09 imobilizado em andamento parado ha mais de 2 anos
registrar('T09_ANDAMENTO_PARADO_2A', base[(base.classe_ne7 == 'IMOBILIZADO_EM_ANDAMENTO') & (base.data_ultima_alteracao < pd.Timestamp('2024-01-01'))],
          'Ordem em andamento sem movimentacao desde antes de 01/01/2024 (transferencia tardia / impairment)')
# T10 data de aquisicao anterior a constituicao da companhia (21/07/1999) em capex proprio
registrar('T10_DATA_ANTERIOR_CONSTITUICAO', base[(base.origem == 'CAPEX_PROPRIO') & (base.data_aquisicao < pd.Timestamp('1999-07-21'))],
          'Data de aquisicao anterior a 21/07/1999', valor=0)
excecoes = pd.concat(exc, ignore_index=True)
excecoes.to_csv(f'{RES}/EXCECOES_TESTES_POPULACAO.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
resumo_testes = excecoes.groupby('teste').agg(qtd=('id_registro', 'count'), valor_em_risco=('valor_em_risco', 'sum')).reset_index()
print(resumo_testes)

                            teste  qtd  valor_em_risco
0                 T01_DUPLICIDADE  160      6737989.18
1                       T02_CORTE   46      2502284.36
2                T03_VCL_NEGATIVO   40      -530782.38
3       T04_RECALCULO_DEPRECIACAO  115      -444915.66
4           T05_BAIXADO_COM_SALDO   63      2553787.67
5      T06_ALMOX_SEM_MOVIMENTO_5A   95       281273.70
6               T07_MAKER_CHECKER  627            0.00
7        T08_CAMPO_NAO_CADASTRADO   35      1234406.07
8         T09_ANDAMENTO_PARADO_2A   45      1134209.97
9  T10_DATA_ANTERIOR_CONSTITUICAO   30            0.00


In [19]:
# =============================================================================
# (3) MUS - Amostragem de unidades monetarias (Gramling, cap. 8; NBC TA 530)
# Afirmacoes: EXISTENCIA e AVALIACAO (teste de superavaliacao). Unidade amostral = R$ 1 do valor contabil liquido.
# Populacao: itens com VCL > 0. Itens com VCL <= 0 sao tratados a parte (T03), como recomenda o material.
# =============================================================================
RD = PARAM['RA'] / (PARAM['RI'] * PARAM['RC'])
col = TAB87[TAB87.RD <= RD + 1e-12].RD.max()          # se o RD calculado nao estiver na tabela, usa a coluna imediatamente inferior (conservador)
FC = float(TAB87.loc[TAB87.RD == col, 'FC'].iloc[0]); FEE = float(TAB87.loc[TAB87.RD == col, 'FEE'].iloc[0]); FATORES = INCR[col]
I = (PARAM['IT'] - PARAM['IE'] * FEE) / FC                                # intervalo de amostragem
pop = base[base.valor_contabil_liquido > 0].copy().sort_values('id_registro').reset_index(drop=True)   # ordem da populacao = id do registro
VC_POP = pop.valor_contabil_liquido.sum()
n_max = VC_POP / I
rng = np.random.default_rng(SEED_MUS); inicio = float(rng.uniform(1, I))  # ponto de partida aleatorio entre 1 e I
pop['acumulado'] = pop.valor_contabil_liquido.cumsum()
pontos = inicio + I * np.arange(int(np.floor((VC_POP - inicio) / I)) + 1)
pos = np.searchsorted(pop.acumulado.values, pontos)                        # item cujo acumulado alcanca o ponto
sel = pd.DataFrame({'ponto_selecao': pontos, 'pos': pos}).groupby('pos').agg(pontos_no_item=('ponto_selecao', 'count'), primeiro_ponto=('ponto_selecao', 'min')).reset_index()
amostra = pop.iloc[sel.pos].reset_index(drop=True).join(sel[['pontos_no_item', 'primeiro_ponto']])
amostra['estrato'] = np.where(amostra.valor_contabil_liquido >= I, 'SUPERIOR (VC >= I)', 'INFERIOR')
# Inspecao documental SIMULADA somente dos itens selecionados
insp = pd.read_csv(f'{B}/INSPECAO_DOCUMENTAL_SIMULADA.csv.gz', sep=';', decimal=',')
amostra = amostra.merge(insp[['id_registro', 'documento_localizado', 'data_documento', 'valor_liquido_suportado', 'observacao_inspecao']], on='id_registro', how='left')
amostra['valor_auditado'] = amostra.valor_liquido_suportado
amostra['incorrecao'] = (amostra.valor_contabil_liquido - amostra.valor_auditado).round(2)
amostra['tainting'] = np.where(amostra.estrato == 'INFERIOR', amostra.incorrecao / amostra.valor_contabil_liquido, np.nan)
amostra['incorrecao_projetada'] = np.where(amostra.estrato == 'INFERIOR', amostra.tainting * I, amostra.incorrecao)
erros = amostra[amostra.incorrecao > 0.005]
inf = erros[erros.estrato == 'INFERIOR'].sort_values('tainting', ascending=False).reset_index(drop=True)
fat = [FATORES[i] if i < len(FATORES) else FATORES[-1] for i in range(len(inf))]   # >10 erros: usa o ultimo fator da tabela (premissa documentada)
inf['fator_incremental'] = fat; inf['ajuste_incremental'] = inf.tainting * inf.fator_incremental * I
PB = FC * I
IMP = erros.incorrecao_projetada.sum()
IA = inf.ajuste_incremental.sum()
LSI = PB + IMP + IA
sub = amostra[amostra.incorrecao < -0.005]
MUS = dict(RD=RD, coluna_tabela=col, FC=FC, FEE=FEE, IT=PARAM['IT'], IE=PARAM['IE'], I=I, VC_populacao=VC_POP, n_populacao=len(pop), n_max=n_max,
           ponto_partida=inicio, n_pontos=len(pontos), n_itens_distintos=len(amostra), n_estrato_superior=int((amostra.estrato != 'INFERIOR').sum()),
           valor_amostra=amostra.valor_contabil_liquido.sum(), n_erros=len(erros), n_erros_inferior=len(inf), PB=PB, IMP=IMP, IA=IA, LSI=LSI,
           conclusao='ACEITAR (LSI <= IT)' if LSI <= PARAM['IT'] else 'NAO ACEITAR SEM TRABALHO ADICIONAL (LSI > IT)', n_subavaliacoes=len(sub))
print(json.dumps({k: (round(v, 2) if isinstance(v, float) else v) for k, v in MUS.items()}, ensure_ascii=False))

{"RD": 0.05, "coluna_tabela": 0.05, "FC": 3.0, "FEE": 1.6, "IT": 20475000.0, "IE": 2047500.0, "I": 5733000.0, "VC_populacao": 6171370101.38, "n_populacao": 517770, "n_max": 1076.46, "ponto_partida": 3429906.84, "n_pontos": 1076, "n_itens_distintos": 772, "n_estrato_superior": 56, "valor_amostra": 2348534050.85, "n_erros": 5, "n_erros_inferior": 5, "PB": 17199000.0, "IMP": 21426739.36, "IA": 11756568.66, "LSI": 50382308.03, "conclusao": "NAO ACEITAR SEM TRABALHO ADICIONAL (LSI > IT)", "n_subavaliacoes": 0}


In [20]:
# =============================================================================
# (4) COMPARACAO - erros inseridos x detectados (o LOG so e lido AGORA)
# =============================================================================
lg = pd.read_csv(f'{B}/controle_erros/LOG_ERROS_INSERIDOS.csv', sep=';', decimal=',', encoding='utf-8-sig')
MAPA = {'E01': 'T01_DUPLICIDADE', 'E02': 'T02_CORTE', 'E03': 'T03_VCL_NEGATIVO', 'E04': 'T04_RECALCULO_DEPRECIACAO', 'E05': 'T05_BAIXADO_COM_SALDO',
        'E06': 'T06_ALMOX_SEM_MOVIMENTO_5A', 'E07': 'T07_MAKER_CHECKER', 'E08': 'T08_CAMPO_NAO_CADASTRADO', 'E09': 'MUS (inspecao documental)',
        'E10': 'T09_ANDAMENTO_PARADO_2A', 'E11': 'T10_DATA_ANTERIOR_CONSTITUICAO'}
comp = []
for tipo, teste in MAPA.items():
    ins = set(lg[lg.tipo_erro == tipo].id_registro)
    det = set(amostra[amostra.incorrecao > 0.005].id_registro) if tipo == 'E09' else set(excecoes[excecoes.teste == teste].id_registro)
    vp = ins & det
    outros_tipos = det - ins
    comp.append(dict(tipo_erro=tipo, descricao=lg[lg.tipo_erro == tipo].descricao.iloc[0], teste=teste, inseridos=len(ins), detectados_pelo_teste=len(det),
                     verdadeiros_positivos=len(vp), nao_detectados=len(ins - det),
                     detectados_de_outro_tipo_inserido=len(outros_tipos & set(lg.id_registro)), falsos_positivos=len(outros_tipos - set(lg.id_registro)),
                     taxa_deteccao=len(vp) / len(ins) if ins else np.nan, efeito_inserido=lg[lg.tipo_erro == tipo]['efeito_monetario_R$'].sum()))
comp = pd.DataFrame(comp)
amostra['erro_inserido'] = amostra.id_registro.map(lg.groupby('id_registro').tipo_erro.apply(lambda s: ','.join(sorted(set(s)))))
total_inserido_superaval = lg.loc[lg['efeito_monetario_R$'] > 0, 'efeito_monetario_R$'].sum()
VAL = dict(incorrecao_projetada_MUS=IMP, LSI=LSI, superavaliacao_total_inserida=total_inserido_superaval,
           LSI_cobre_erro_real='SIM' if LSI >= total_inserido_superaval else 'NAO')
print(comp[['tipo_erro', 'inseridos', 'verdadeiros_positivos', 'falsos_positivos', 'taxa_deteccao']]); print(VAL)

   tipo_erro  inseridos  verdadeiros_positivos  falsos_positivos  \
0        E01        160                    160                 0   
1        E02         46                     46                 0   
2        E03         40                     40                 0   
3        E04         70                     70                 0   
4        E05         63                     63                 0   
5        E06         95                     95                 0   
6        E07        620                    620                 0   
7        E08         35                     35                 0   
8        E09         97                      2                 0   
9        E10         45                     45                 0   
10       E11         30                     30                 0   

    taxa_deteccao  
0        1.000000  
1        1.000000  
2        1.000000  
3        1.000000  
4        1.000000  
5        1.000000  
6        1.000000  
7        1.000000  
8  

In [21]:
# =============================================================================
# Saidas: planilhas editaveis (com formulas) e CSVs
# =============================================================================
HEAD = PatternFill('solid', fgColor='1F4E78'); thin = Side(style='thin', color='BFBFBF'); BD = Border(top=thin, bottom=thin, left=thin, right=thin)
SINT = PatternFill('solid', fgColor='FFF2CC')
RODAPE = 'DADOS SINTETICOS - procedimento SIMULADO PARA FINS DIDATICOS | Preparador: matricula 211059370 | Data: 23/10/2026 | Revisao: ______ (nao preenchido)'
def cab(ws, r, cols, w=None):
    for j, c in enumerate(cols, 1):
        x = ws.cell(row=r, column=j, value=c); x.font = Font(bold=True, color='FFFFFF'); x.fill = HEAD; x.alignment = Alignment(wrap_text=True); x.border = BD
    if w:
        for j, k in enumerate(w, 1): ws.column_dimensions[get_column_letter(j)].width = k
def tit(ws, t):
    ws['A1'] = t; ws['A1'].font = Font(bold=True, size=13); ws['A2'] = RODAPE; ws['A2'].font = Font(italic=True, size=9, color='9C5700'); ws['A2'].fill = SINT
def df_to(ws, df, r0, nf=None):
    for i, row in enumerate(df.itertuples(index=False), r0):
        for j, v in enumerate(row, 1):
            if isinstance(v, (np.floating, float)) and np.isnan(v): v = None
            elif isinstance(v, np.generic): v = v.item()
            elif isinstance(v, pd.Timestamp): v = v.date()
            c = ws.cell(row=i, column=j, value=v); c.border = BD
            if nf and j in nf: c.number_format = nf[j]
    return r0 + len(df) - 1
NUM = '#,##0.00;(#,##0.00);-'; PCT = '0.00%'

# ---- CONC_BASE_RECV3.xlsx
wb = Workbook(); ws = wb.active; ws.title = 'Conc_classe'
tit(ws, 'Conciliacao: base simulada x NE 7.1 (F01 p.68-69) - R$ (centavos)')
cols = ['Classe NE 7', 'Registros', 'Custo base (ctr)', 'Custo NE 7 (ctr)', 'Dif.', 'Custo base (cons)', 'Custo NE 7 (cons)', 'Dif.',
        'Acum. base (ctr)', 'Acum. NE 7 (ctr)', 'Dif.', 'Acum. base (cons)', 'Acum. NE 7 (cons)', 'Dif.', 'Deprec. 2025 base (cons)', 'Deprec. 2025 NE 7 (cons)', 'Dif.']
cab(ws, 4, cols, [32, 10] + [17] * 15)
for i, r in enumerate(conc.itertuples(index=False), 5):
    vals = [r.classe_ne7, r.n_registros, r.custo_base_ctr, r.custo_ne7_ctr, f'=C{i}-D{i}', r.custo_base_cons, r.custo_ne7_cons, f'=F{i}-G{i}',
            r.acum_base_ctr, r.acum_ne7_ctr, f'=I{i}-J{i}', r.acum_base_cons, r.acum_ne7_cons, f'=L{i}-M{i}', r.dep2025_base_cons, r.dep2025_ne7_cons, f'=O{i}-P{i}']
    for j, v in enumerate(vals, 1):
        c = ws.cell(row=i, column=j, value=v); c.border = BD; c.number_format = NUM if j > 2 else 'General'
t = 5 + len(conc)
ws.cell(row=t, column=1, value='TOTAL IMOBILIZADO').font = Font(bold=True)
for j in range(2, 18):
    L_ = get_column_letter(j); ws.cell(row=t, column=j, value=f'=SUM({L_}5:{L_}{t-1})').number_format = NUM
ws.cell(row=t + 1, column=1, value='Valor liquido consolidado - base'); ws.cell(row=t + 1, column=6, value=f'=F{t}-L{t}').number_format = NUM
ws.cell(row=t + 2, column=1, value='Valor liquido consolidado - CVM 1.02.03 (F03) / BP (F01 p.51, R$ mil x 1.000)'); ws.cell(row=t + 2, column=6, value=6170841000).number_format = NUM
ws.cell(row=t + 3, column=1, value='Diferenca (arredondamento da DF em R$ mil)'); ws.cell(row=t + 3, column=6, value=f'=F{t+1}-F{t+2}').number_format = NUM
notas = ['Explicacao das diferencas: (1) A DF e publicada em R$ mil (arredondada); a base, em centavos. A geracao introduz, por classe/entidade, diferenca simulada de',
         'arredondamento de no maximo R$ 240, de modo que cada subtotal arredondado para R$ mil coincide com a NE 7.1. (2) A depreciacao do exercicio dos itens LINEARES',
         'resulta das datas simuladas e excede a NE 7.1 (a base foi calibrada para os SALDOS e para a deplecao UoP); a diferenca nao afeta a conciliacao do saldo e e uma',
         'limitacao do dado sintetico. (3) Intangivel (software, R$ 83.384 mil) fora da base: a conta escolhida e o imobilizado (98,7% da rubrica Imobilizado e intangivel).',
         '(4) Os erros inseridos estao DENTRO do saldo contabil (a base reproduz um razao que contem distorcoes simuladas); por isso o total da base continua igual a DF.']
for k, n in enumerate(notas): ws.cell(row=t + 5 + k, column=1, value=n)
ws2 = wb.create_sheet('Balancete_plano_simulado'); tit(ws2, 'Balancete do imobilizado por conta do PLANO DE CONTAS SIMULADO')
cab(ws2, 4, ['Conta', 'Descricao (simulada)', 'Custo', 'Deprec. acumulada', 'Valor liquido', 'Registros'], [16, 70, 18, 18, 18, 10])
e = df_to(ws2, balancete, 5, {3: NUM, 4: NUM, 5: NUM})
ws2.cell(row=e + 1, column=2, value='TOTAL').font = Font(bold=True)
for j in (3, 4, 5, 6): ws2.cell(row=e + 1, column=j, value=f'=SUM({get_column_letter(j)}5:{get_column_letter(j)}{e})').number_format = NUM
wb.save('planilhas/CONC_BASE_RECV3.xlsx')

# ---- TESTES_BASE_RECV3.xlsx (resumo dos testes de populacao, MUS e comparacao)
wb = Workbook(); ws = wb.active; ws.title = 'Resumo_testes_populacao'
tit(ws, 'Testes em 100% da populacao (T01-T10) - resumo')
cab(ws, 4, ['Teste', 'Qtd. excecoes', 'Valor em risco (R$)', 'Afirmacao', 'Norma'], [34, 14, 20, 30, 26])
AFIRM = {'T01_DUPLICIDADE': ('Existencia', 'NBC TA 500 / 240'), 'T02_CORTE': ('Existencia / corte', 'NBC TA 500 / 330'), 'T03_VCL_NEGATIVO': ('Avaliacao', 'NBC TA 500 / 520'),
         'T04_RECALCULO_DEPRECIACAO': ('Avaliacao (exatidao)', 'NBC TA 500 A21 (recalculo)'), 'T05_BAIXADO_COM_SALDO': ('Existencia', 'NBC TA 500'),
         'T06_ALMOX_SEM_MOVIMENTO_5A': ('Avaliacao (recuperabilidade)', 'NBC TA 540 / 501'), 'T07_MAKER_CHECKER': ('Controle - segregacao', 'NBC TA 315 / 265'),
         'T08_CAMPO_NAO_CADASTRADO': ('Existencia / classificacao', 'NBC TA 500 / 240'), 'T09_ANDAMENTO_PARADO_2A': ('Avaliacao / classificacao', 'NBC TA 540'),
         'T10_DATA_ANTERIOR_CONSTITUICAO': ('Integridade cadastral', 'NBC TA 500 (confiabilidade dos dados)')}
for i, r in enumerate(resumo_testes.itertuples(index=False), 5):
    for j, v in enumerate([r.teste, int(r.qtd), float(r.valor_em_risco), *AFIRM[r.teste]], 1):
        c = ws.cell(row=i, column=j, value=v); c.border = BD; c.number_format = NUM if j == 3 else 'General'
ws3 = wb.create_sheet('Excecoes_detalhe'); tit(ws3, 'Detalhe das excecoes (todas)')
cab(ws3, 4, list(excecoes.columns), [30, 16, 30, 18, 18, 70]); df_to(ws3, excecoes, 5, {4: NUM, 5: NUM})

# MUS - parametros com formulas
wm = wb.create_sheet('MUS_parametros'); tit(wm, 'MUS - parametros e intervalo (Gramling cap. 8; NBC TA 530) - formulas editaveis')
cab(wm, 4, ['Parametro', 'Valor', 'Fonte / formula'], [52, 22, 80])
linhas = [('Materialidade global provisoria - MG (R$)', PARAM['MG'], 'PT 320: 1% da receita liquida consolidada 2025 (R$ 3.157.609 mil) = 31.576 mil, arredondado p/ baixo (NBC TA 320 itens 10, A5, A8)'),
          ('% da MG para execucao', PARAM['pct_ME'], 'Julgamento (NBC TA 320 item 11 e A13): primeira auditoria + inconsistencias de divulgacao + risco de impairment'),
          ('Materialidade para execucao - ME (R$)', '=B5*B6', 'MG x %'),
          ('Incorrecao toleravel - IT (R$)', '=B7', 'IT = ME (sem materialidade especifica para o imobilizado)'),
          ('% IT para incorrecao esperada', PARAM['pct_IE'], 'Premissa do auditor (sem historico: primeira auditoria)'),
          ('Incorrecao esperada - IE (R$)', '=B8*B9', 'IT x %'),
          ('Risco de auditoria - RA', PARAM['RA'], 'Gramling: RA = RI x RC x RD'), ('Risco inerente - RI', PARAM['RI'], 'Maximo: estimativas e julgamentos (PT 315)'),
          ('Risco de controle - RC', PARAM['RC'], 'Maximo: controles nao testados (sem acesso)'), ('Risco de deteccao - RD', '=B11/(B12*B13)', 'RD = RA / (RI x RC)'),
          ('Coluna da Ilustracao 8.7 utilizada', col, 'RD da tabela imediatamente inferior ou igual ao calculado'),
          ('Fator de confiabilidade - FC', FC, 'Ilustracao 8.7'), ('Fator de expansao de erro - FEE', FEE, 'Ilustracao 8.7'),
          ('Intervalo de amostragem - I (R$)', '=(B8-B10*B17)/B16', 'I = (IT - IE x FEE) / FC'),
          ('Valor contabil da populacao (VCL > 0) (R$)', VC_POP, f'{len(pop):,} itens; itens com VCL <= 0 testados a parte (T03)'.replace(',', '.')),
          ('Tamanho maximo da amostra - n', '=B19/B18', 'n = valor contabil / I'),
          ('Ponto de partida aleatorio (R$)', inicio, f'Gerador numpy, semente {SEED_MUS} (entre 1 e I)'),
          ('Pontos de selecao', len(pontos), 'inicio + k x I'), ('Itens distintos selecionados', len(amostra), 'itens com mais de um ponto contam uma vez'),
          ('Itens do estrato superior (VC >= I)', MUS['n_estrato_superior'], 'selecao certa (100%)')]
for i, (a, v, f) in enumerate(linhas, 5):
    wm.cell(row=i, column=1, value=a); c = wm.cell(row=i, column=2, value=v); wm.cell(row=i, column=3, value=f)
    c.number_format = '0.00%' if i in (6, 9, 11, 12, 13, 14, 15) else ('0.00' if i in (16, 17, 20) else NUM)
    for j in (1, 2, 3): wm.cell(row=i, column=j).border = BD
wm.cell(row=27, column=1, value='Ilustracao 8.7 (Gramling) - fatores').font = Font(bold=True)
cab(wm, 28, ['Risco de deteccao', 'FC', 'FEE'] + [f'Incr. {k}' for k in range(1, 11)])
for i, rd in enumerate(TAB87.RD, 29):
    for j, v in enumerate([rd, float(TAB87.FC[i - 29]), float(TAB87.FEE[i - 29])] + INCR[rd], 1): wm.cell(row=i, column=j, value=v).border = BD

# MUS - selecao e resultado da inspecao
wa = wb.create_sheet('MUS_amostra'); tit(wa, 'MUS - itens selecionados e resultado da inspecao documental SIMULADA')
cols_a = ['id_registro', 'classe_ne7', 'conta_custo', 'campo', 'tipo_item', 'estrato', 'pontos_no_item', 'primeiro_ponto', 'acumulado', 'valor_contabil_liquido',
          'documento_localizado', 'data_documento', 'valor_auditado', 'incorrecao', 'tainting', 'incorrecao_projetada', 'observacao_inspecao']
cab(wa, 4, cols_a, [16, 28, 14, 22, 30, 18, 8, 16, 18, 16, 8, 12, 16, 14, 10, 16, 50])
I_REF = "MUS_parametros!$B$18"
for i, r in enumerate(amostra[cols_a].itertuples(index=False), 5):
    for j, v in enumerate(r, 1):
        if isinstance(v, (np.floating, float)) and np.isnan(v): v = None
        elif isinstance(v, np.generic): v = v.item()
        c = wa.cell(row=i, column=j, value=v); c.border = BD
        if j in (8, 9, 10, 13, 14, 16): c.number_format = NUM
    wa.cell(row=i, column=14, value=f'=J{i}-M{i}').number_format = NUM
    wa.cell(row=i, column=15, value=f'=IF(F{i}="INFERIOR",N{i}/J{i},"")').number_format = PCT
    wa.cell(row=i, column=16, value=f'=IF(F{i}="INFERIOR",O{i}*{I_REF},N{i})').number_format = NUM
last_a = 4 + len(amostra)

# MUS - avaliacao com formulas (LSI)
wv = wb.create_sheet('MUS_avaliacao'); tit(wv, 'MUS - avaliacao: LSI = precisao basica + incorrecao mais provavel + ajuste incremental')
cab(wv, 4, ['Componente', 'Valor (R$)', 'Formula / explicacao'], [52, 22, 90])
aval = [('Precisao basica (PB)', f'=MUS_parametros!B16*{I_REF}', 'FC x I (LSI se nenhum erro fosse encontrado)'),
        ('Incorrecao mais provavel - estrato inferior (projetada)', f'=SUMIFS(MUS_amostra!P5:P{last_a},MUS_amostra!F5:F{last_a},"INFERIOR",MUS_amostra!N5:N{last_a},">0")', 'soma de tainting x I'),
        ('Incorrecao - estrato superior (real, sem projecao)', f'=SUMIFS(MUS_amostra!N5:N{last_a},MUS_amostra!F5:F{last_a},"<>INFERIOR",MUS_amostra!N5:N{last_a},">0")', 'diferenca efetiva'),
        ('Incorrecao mais provavel (IMP)', '=B6+B7', 'projetada + real'),
        ('Ajuste incremental (IA)', '=SUM(E13:E200)', 'taintings em ordem decrescente x fator incremental x I (tabela abaixo)'),
        ('LIMITE SUPERIOR DE INCORRECAO (LSI)', '=B5+B8+B9', 'PB + IMP + IA'),
        ('Incorrecao toleravel (IT)', '=MUS_parametros!B8', ''), ('Conclusao', '=IF(B10<=B11,"ACEITAR: LSI <= IT","NAO ACEITAR SEM TRABALHO ADICIONAL: LSI > IT")', 'regra de decisao do material (pag. 302)')]
for i, (a, f, x) in enumerate(aval, 5):
    wv.cell(row=i, column=1, value=a).border = BD; c = wv.cell(row=i, column=2, value=f); c.border = BD; c.number_format = NUM; wv.cell(row=i, column=3, value=x).border = BD
wv['B12'].number_format = 'General'; wv['A10'].font = Font(bold=True); wv['B10'].font = Font(bold=True)
cab(wv, 12, ['Ordem', 'id_registro', 'Tainting', 'Fator incremental (col. RD)', 'Ajuste incremental (R$)'])
for k, r in enumerate(inf.itertuples(index=False), 13):
    wv.cell(row=k, column=1, value=k - 12); wv.cell(row=k, column=2, value=r.id_registro)
    wv.cell(row=k, column=3, value=float(r.tainting)).number_format = PCT; wv.cell(row=k, column=4, value=float(r.fator_incremental))
    wv.cell(row=k, column=5, value=f'=C{k}*D{k}*{I_REF}').number_format = NUM
nota_row = 14 + len(inf)
for k, n in enumerate(['Premissa: a Ilustracao 8.7 traz fatores ate o 10o erro; para erros adicionais foi usado o ultimo fator da coluna (premissa conservadora documentada).',
                       'Subavaliacoes (valor auditado > contabil) nao entram no LSI de superavaliacao; sao avaliadas a parte no PT 450.',
                       'Resultado calculado em Python (conferencia): ' + f'PB={PB:,.2f}; IMP={IMP:,.2f}; IA={IA:,.2f}; LSI={LSI:,.2f}; IT={PARAM["IT"]:,.2f}.']):
    wv.cell(row=nota_row + k, column=1, value=n)

wc = wb.create_sheet('Inseridos_x_detectados'); tit(wc, 'Comparacao: erros INSERIDOS (log de controle) x DETECTADOS pelos testes')
cab(wc, 4, list(comp.columns), [8, 60, 30, 10, 12, 12, 12, 14, 12, 12, 18])
df_to(wc, comp, 5, {10: '0.0%', 11: NUM})
r_ = 7 + len(comp)
for k, n in enumerate([f'MUS: incorrecao mais provavel projetada = R$ {IMP:,.2f}; LSI = R$ {LSI:,.2f}.',
                       f'Superavaliacao total efetivamente inserida na base (todos os tipos monetarios) = R$ {total_inserido_superaval:,.2f}.',
                       f'O LSI {"cobre" if LSI >= total_inserido_superaval else "NAO cobre"} a superavaliacao real inserida - validacao didatica do metodo.',
                       'Os testes T01-T10 foram executados antes da leitura do log (as cegas); o log foi usado apenas nesta aba.']):
    wc.cell(row=r_ + k, column=1, value=n)
wb.save('planilhas/TESTES_BASE_RECV3.xlsx')

amostra.to_csv(f'{RES}/MUS_AMOSTRA_SELECIONADA.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
comp.to_csv(f'{RES}/COMPARATIVO_INSERIDOS_DETECTADOS.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
json.dump({'parametros': PARAM, 'mus': MUS, 'validacao': VAL}, open(f'{RES}/RESULTADOS_MUS.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=float)
print('ok')

ok


## D1. SIMULAÇÃO da base complementar de folha e benefícios a empregados
> 🟨 **SEÇÃO DE SIMULAÇÃO**

Código-fonte: `codigo/gerar_base_folha.py` (o mesmo código, dividido em células).

In [22]:
# -*- coding: utf-8 -*-
# =============================================================================
# P2 - AUDITORIA 1 (UnB 2026.2) - PetroReconcavo S.A. (RECV3) - exercicio 2025
# Preparador: matricula 211059370
#
# >>> SECAO DE SIMULACAO - BASE COMPLEMENTAR (folha e beneficios a empregados) <<<
# Complementa a base obrigatoria (imobilizado). Atende a orientacao de aula sobre beneficios a empregados:
# circulante (vence ate 31/12/2026) x nao circulante (a partir de 01/01/2027); riscos de empregado ficticio,
# cadastro duplicado e pagamento sem vinculo; controle maker-checker e trilha de alteracoes.
#
# !!! DADOS SINTETICOS. Nenhum empregado, CPF ou conta bancaria e real. !!!
# Ancoras REAIS: quadro de 1.752 empregados (249 em lideranca) em 31/12/2025 - FRE 2026 item 10.1A (F13);
# obrigacoes sociais e trabalhistas consolidadas R$ 88.753 mil (CVM 2.01.01, F03);
# despesa de pessoal consolidada 2025 R$ 284.002 mil (F01 NE 19, p.93);
# provisao para PLR inferida do IR diferido: R$ 11.054 mil / 34% = R$ 32.512 mil (F01 NE 10.2, p.77) - estimativa do preparador.
# Uso:  python codigo/gerar_base_folha.py
# =============================================================================
import os, json
import numpy as np
import pandas as pd

SEED = 211059370 + 7
rng = np.random.default_rng(SEED)
OUT = 'base_simulada/folha'; CTRL = 'base_simulada/controle_erros'
os.makedirs(OUT, exist_ok=True); os.makedirs(CTRL, exist_ok=True)
DB = pd.Timestamp('2025-12-31')
ATIVOS, LIDER = 1752, 249
OBRIG_TRAB = 88_753_000.00                  # CVM 2.01.01 consolidado (R$)
DESP_PESSOAL = 284_002_000.00               # NE 19 consolidado (R$)
PLR_PROV = round(11_054_000 / 0.34, 2)      # inferencia a partir do IR diferido (NE 10.2)
ENC_INSS, ENC_FGTS = 0.288, 0.08            # encargos patronais simulados (INSS+RAT+terceiros; FGTS)

In [23]:
# ---------------------------------------------------------------- 1) cadastro de empregados
n_desl = 190                                 # desligados ao longo de 2025 (turnover simulado)
N = ATIVOS + n_desl
lider = np.zeros(N, bool); lider[rng.choice(ATIVOS, LIDER, replace=False)] = True; lider[ATIVOS:] = rng.random(n_desl) < 0.08
polos = rng.choice(['SEDE SALVADOR', 'BAHIA - REMANSO', 'BAHIA - MIRANGA', 'BAHIA - SAO ROQUE (UTG)', 'POTIGUAR - MOSSORO', 'TIETA'], N, p=[.14, .22, .22, .08, .31, .03])
cad = pd.DataFrame({
    'matricula': [f'M{i:05d}' for i in range(1, N + 1)],
    'entidade': np.where(polos == 'TIETA', 'SPE_TIETA', 'PETRORECONCAVO'),
    'cpf_simulado': [f'SIM-{x:09d}' for x in pd.unique(rng.integers(100_000_000, 999_999_999, N * 2))[:N]],
    'nivel': np.where(lider, 'LIDERANCA', 'NAO_LIDERANCA'),
    'lotacao': polos,
    'data_admissao': pd.Timestamp('2000-03-01') + pd.to_timedelta((rng.beta(2.8, 1.3, N) * (pd.Timestamp('2025-11-30') - pd.Timestamp('2000-03-01')).days).astype(int), unit='D'),
    'data_desligamento': pd.NaT,
    'salario_base_mensal': np.where(lider, rng.lognormal(np.log(19000), 0.45, N), rng.lognormal(np.log(6200), 0.40, N)),
    'banco_agencia_conta': [f'{b}-{a:04d}-{c:07d}' for b, a, c in zip(rng.choice(['001', '104', '237', '341', '033', '756'], N), rng.integers(1, 9999, N), rng.integers(1, 9_999_999, N))],
    'contrato_trabalho_arquivado': 'S',
})
cad.loc[ATIVOS:, 'data_desligamento'] = pd.Timestamp('2025-01-10') + pd.to_timedelta(rng.integers(0, 345, n_desl), unit='D')
cad.loc[ATIVOS:, 'data_admissao'] = np.minimum(cad.loc[ATIVOS:, 'data_admissao'], cad.loc[ATIVOS:, 'data_desligamento'] - pd.Timedelta(days=200))
usr = [f'RH{i:02d}' for i in range(1, 9)]
cad['usuario_cadastro'] = rng.choice(usr, N)
cad['usuario_aprovacao'] = [usr[(usr.index(u) + int(o)) % len(usr)] for u, o in zip(cad.usuario_cadastro, rng.integers(1, len(usr), N))]
cad['data_ultima_alteracao_salarial'] = (cad.data_admissao + pd.to_timedelta(rng.integers(0, 900, N), unit='D')).clip(upper=DB)
cad['pct_ultimo_reajuste'] = np.round(rng.normal(0.055, 0.025, N).clip(0, 0.18), 4)
cad['aprovador_reajuste'] = rng.choice(['GERENCIA_RH', 'DIRETORIA_PESSOAS'], N, p=[.85, .15])

In [24]:
# ---------------------------------------------------------------- 2) folha de pagamentos 2025 (competencias jan-dez + 13o + ferias + PLR)
linhas = []
for comp in range(1, 13):
    ini = pd.Timestamp(2025, comp, 1); fim = ini + pd.offsets.MonthEnd(0)
    ativo = (cad.data_admissao <= fim) & (cad.data_desligamento.isna() | (cad.data_desligamento >= ini))
    c = cad[ativo]
    adic = rng.choice([0.0, 0.30], len(c), p=[.45, .55])                 # adicional de periculosidade (campo)
    he = rng.gamma(1.2, 0.04, len(c))
    for rub, mult in (('SALARIO', 1.0), ('ADICIONAL_PERICULOSIDADE', adic), ('HORAS_EXTRAS', he)):
        v = c.salario_base_mensal.values * mult
        m = v > 0
        linhas.append(pd.DataFrame({'competencia': f'2025-{comp:02d}', 'matricula': c.matricula.values[m], 'rubrica': rub, 'valor_bruto': v[m],
                                    'data_pagamento': (fim + pd.Timedelta(days=5)).date(), 'conta_credito': c.banco_agencia_conta.values[m]}))
ativos_dez = cad[cad.data_desligamento.isna()]
linhas.append(pd.DataFrame({'competencia': '2025-13', 'matricula': ativos_dez.matricula, 'rubrica': '13O_SALARIO', 'valor_bruto': ativos_dez.salario_base_mensal,
                            'data_pagamento': pd.Timestamp('2025-12-19').date(), 'conta_credito': ativos_dez.banco_agencia_conta}))
fer = cad.sample(frac=0.92, random_state=SEED)
linhas.append(pd.DataFrame({'competencia': rng.choice([f'2025-{m:02d}' for m in range(1, 13)], len(fer)), 'matricula': fer.matricula, 'rubrica': 'FERIAS_+1/3',
                            'valor_bruto': fer.salario_base_mensal * 4 / 3, 'data_pagamento': None, 'conta_credito': fer.banco_agencia_conta}))
plr = cad[cad.data_admissao < pd.Timestamp('2025-01-01')]
linhas.append(pd.DataFrame({'competencia': '2025-03', 'matricula': plr.matricula, 'rubrica': 'PLR_2024_PAGA', 'valor_bruto': plr.salario_base_mensal * rng.uniform(1.2, 2.6, len(plr)),
                            'data_pagamento': pd.Timestamp('2025-03-28').date(), 'conta_credito': plr.banco_agencia_conta}))
rescis = cad[cad.data_desligamento.notna()]
linhas.append(pd.DataFrame({'competencia': rescis.data_desligamento.dt.strftime('%Y-%m'), 'matricula': rescis.matricula, 'rubrica': 'VERBAS_RESCISORIAS',
                            'valor_bruto': rescis.salario_base_mensal * rng.uniform(1.5, 4.0, len(rescis)), 'data_pagamento': (rescis.data_desligamento + pd.Timedelta(days=10)).dt.date,
                            'conta_credito': rescis.banco_agencia_conta}))
folha = pd.concat(linhas, ignore_index=True)
folha['data_pagamento'] = folha.data_pagamento.fillna(pd.Timestamp('2025-06-30').date())

# calibracao: custo total do ano (bruto + encargos + beneficios + variacao de provisoes) = despesa de pessoal da NE 19
BENEF = 0.115                                # beneficios (saude, alimentacao, previdencia) sobre a folha bruta - premissa simulada
bruto = folha.valor_bruto.sum()
k = (DESP_PESSOAL - PLR_PROV * (1 + ENC_INSS + ENC_FGTS)) / (bruto * (1 + ENC_INSS + ENC_FGTS + BENEF))
folha['valor_bruto'] = (folha.valor_bruto * k).round(2)
cad['salario_base_mensal'] = (cad.salario_base_mensal * k).round(2)
folha['encargos_patronais'] = (folha.valor_bruto * (ENC_INSS + ENC_FGTS)).round(2)
folha['beneficios'] = (folha.valor_bruto * BENEF).round(2)

In [25]:
# ---------------------------------------------------------------- 3) obrigacoes trabalhistas em 31/12/2025 (por empregado) - total = CVM 2.01.01
prov = []
ad = cad[cad.data_desligamento.isna()].copy()
sal_dez = ad.salario_base_mensal
meses_fer = rng.integers(0, 12, len(ad)) + 1
for tipo, v, venc in (('SALARIO_DEZ_A_PAGAR', sal_dez * (1 + 0.55 * 0.30), pd.Timestamp('2026-01-05')),
                      ('INSS_FGTS_DEZ_A_RECOLHER', sal_dez * (ENC_INSS + ENC_FGTS), pd.Timestamp('2026-01-20')),
                      ('FERIAS_PROPORCIONAIS_+1/3', sal_dez * 4 / 3 * meses_fer / 12, pd.Timestamp('2026-06-30')),
                      ('ENCARGOS_SOBRE_FERIAS', sal_dez * 4 / 3 * meses_fer / 12 * (ENC_INSS + ENC_FGTS), pd.Timestamp('2026-06-30'))):
    prov.append(pd.DataFrame({'matricula': ad.matricula, 'tipo_obrigacao': tipo, 'valor': np.asarray(v, float), 'data_vencimento': venc}))
w = ad.salario_base_mensal * np.where(ad.nivel == 'LIDERANCA', 2.5, 1.0)
prov.append(pd.DataFrame({'matricula': ad.matricula, 'tipo_obrigacao': 'PLR_2025_PROVISIONADA', 'valor': w / w.sum() * PLR_PROV, 'data_vencimento': pd.Timestamp('2026-03-31')}))
prov = pd.concat(prov, ignore_index=True)
resto = OBRIG_TRAB - prov.valor.sum()
assert resto > 0, resto
n_out = 400; idx = rng.choice(len(ad), n_out, replace=False)
prov = pd.concat([prov, pd.DataFrame({'matricula': ad.matricula.values[idx], 'tipo_obrigacao': 'OUTRAS (ENCARGOS S/ PLR, RESCISOES, BENEFICIOS A PAGAR)',
                                      'valor': rng.dirichlet(np.ones(n_out)) * resto, 'data_vencimento': pd.Timestamp('2026-02-28')})], ignore_index=True)
prov['valor'] = prov.valor.round(2)
prov.loc[prov.valor.idxmax(), 'valor'] += round(OBRIG_TRAB - prov.valor.sum(), 2)   # fecha exatamente em centavos
prov['classificacao_registrada'] = 'CIRCULANTE'

In [26]:
# ---------------------------------------------------------------- 4) erros inseridos (log separado)
LOG = []
def log(tipo, ids, desc, efeito=0.0):
    for i, e in zip(ids, np.broadcast_to(efeito, len(ids))):
        LOG.append({'tipo_erro': tipo, 'chave': i, 'descricao': desc, 'efeito_monetario_R$': float(e)})
# F01 - empregado ficticio: pagamentos para matriculas que nao existem no cadastro
fic = [f'M{N + i:05d}' for i in range(1, 7)]
lf = []
for m in fic:
    s = float(rng.uniform(9000, 16000)); conta = f'341-{rng.integers(1, 9999):04d}-{rng.integers(1, 9_999_999):07d}'
    for comp in range(4, 13):
        lf.append(dict(competencia=f'2025-{comp:02d}', matricula=m, rubrica='SALARIO', valor_bruto=round(s, 2), data_pagamento=(pd.Timestamp(2025, comp, 1) + pd.offsets.MonthEnd(0) + pd.Timedelta(days=5)).date(),
                       conta_credito=conta, encargos_patronais=0.0, beneficios=0.0))
lf = pd.DataFrame(lf); folha = pd.concat([folha, lf], ignore_index=True)
for m in fic: log('F01', [m], 'Empregado ficticio: pagamentos de salario a matricula inexistente no cadastro', lf[lf.matricula == m].valor_bruto.sum())
# F02 - cadastro duplicado: mesmo CPF e/ou mesma conta bancaria para matriculas diferentes
src = rng.choice(len(ad), 9, replace=False)
dst = rng.choice(np.setdiff1d(np.arange(len(ad)), src), 9, replace=False)
for s_, d_ in zip(src[:5], dst[:5]):
    cad.loc[cad.matricula == ad.matricula.iloc[d_], 'cpf_simulado'] = ad.cpf_simulado.iloc[s_]; log('F02', [ad.matricula.iloc[d_]], 'CPF identico ao de outra matricula')
for s_, d_ in zip(src[5:], dst[5:]):
    cad.loc[cad.matricula == ad.matricula.iloc[d_], 'banco_agencia_conta'] = ad.banco_agencia_conta.iloc[s_]
    folha.loc[folha.matricula == ad.matricula.iloc[d_], 'conta_credito'] = ad.banco_agencia_conta.iloc[s_]
    log('F02', [ad.matricula.iloc[d_]], 'Conta bancaria de credito identica a de outra matricula')
# F03 - pagamento apos o desligamento (sem vinculo empregaticio)
dl = cad[cad.data_desligamento.notna() & (cad.data_desligamento < pd.Timestamp('2025-09-01'))].sample(8, random_state=SEED)
for _, r in dl.iterrows():
    comp = r.data_desligamento.month + 2
    lp = dict(competencia=f'2025-{comp:02d}', matricula=r.matricula, rubrica='SALARIO', valor_bruto=r.salario_base_mensal,
              data_pagamento=(pd.Timestamp(2025, comp, 1) + pd.offsets.MonthEnd(0) + pd.Timedelta(days=5)).date(), conta_credito=r.banco_agencia_conta,
              encargos_patronais=round(r.salario_base_mensal * (ENC_INSS + ENC_FGTS), 2), beneficios=0.0)
    folha = pd.concat([folha, pd.DataFrame([lp])], ignore_index=True)
    log('F03', [r.matricula], 'Salario pago apos a data de desligamento', r.salario_base_mensal)
# F04 - cadastro sem aprovacao ou aprovado pelo proprio cadastrador (maker-checker)
ix = rng.choice(len(cad), 30, replace=False)
cad.loc[cad.index[ix[:15]], 'usuario_aprovacao'] = ''
cad.loc[cad.index[ix[15:]], 'usuario_aprovacao'] = cad.loc[cad.index[ix[15:]], 'usuario_cadastro'].values
log('F04', cad.matricula.iloc[ix].tolist(), 'Cadastro de empregado sem aprovacao independente (maker-checker)')
# F05 - reajuste salarial acima de 30% sem aprovacao da diretoria (trilha de alteracoes)
ix = rng.choice(np.where(cad.data_desligamento.isna())[0], 12, replace=False)
cad.loc[cad.index[ix], 'pct_ultimo_reajuste'] = np.round(rng.uniform(0.32, 0.65, 12), 4); cad.loc[cad.index[ix], 'aprovador_reajuste'] = 'GERENCIA_RH'
cad.loc[cad.index[ix], 'data_ultima_alteracao_salarial'] = pd.Timestamp('2025-08-01') + pd.to_timedelta(rng.integers(0, 120, 12), unit='D')
log('F05', cad.matricula.iloc[ix].tolist(), 'Reajuste salarial > 30% sem aprovacao da Diretoria (alcada)')
# F06 - obrigacao com vencimento apos 31/12/2026 registrada como CIRCULANTE (classificacao)
iv = prov[prov.tipo_obrigacao.str.startswith('OUTRAS')].sample(25, random_state=SEED).index
prov.loc[iv, 'tipo_obrigacao'] = 'BONUS DE RETENCAO DIFERIDO'; prov.loc[iv, 'data_vencimento'] = pd.Timestamp('2027-03-31') + pd.to_timedelta(rng.integers(0, 300, 25), unit='D')
log('F06', [f'PROV-{i:06d}' for i in iv], 'Obrigacao com vencimento a partir de 01/01/2027 classificada no circulante', prov.loc[iv, 'valor'].values)

prov.insert(0, 'id_obrigacao', [f'PROV-{i:06d}' for i in prov.index])
for df in (cad, folha, prov): df['natureza_dado'] = 'SINTETICO'
folha.insert(0, 'id_lancamento', [f'FOL-{i:07d}' for i in range(1, len(folha) + 1)])
cad.to_csv(f'{OUT}/CADASTRO_EMPREGADOS_SIMULADO.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig', date_format='%Y-%m-%d')
folha.to_csv(f'{OUT}/FOLHA_PAGAMENTOS_2025_SIMULADA.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
prov.to_csv(f'{OUT}/OBRIGACOES_TRABALHISTAS_31122025_SIMULADAS.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig', date_format='%Y-%m-%d')
pd.DataFrame(LOG).to_csv(f'{CTRL}/LOG_ERROS_INSERIDOS_FOLHA.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
custo = folha.valor_bruto.sum() + folha.encargos_patronais.sum() + folha.beneficios.sum() + PLR_PROV * (1 + ENC_INSS + ENC_FGTS)
meta = dict(cadastro=len(cad), ativos_31122025=int(cad.data_desligamento.isna().sum()), lancamentos_folha=len(folha), obrigacoes=len(prov),
            total_obrigacoes=float(prov.valor.sum()), custo_pessoal_2025_simulado=float(custo), plr_inferida=PLR_PROV, erros=pd.DataFrame(LOG).tipo_erro.value_counts().sort_index().to_dict())
json.dump(meta, open(f'{OUT}/METADADOS_FOLHA.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(meta)

{'cadastro': 1942, 'ativos_31122025': 1752, 'lancamentos_folha': 61426, 'obrigacoes': 9160, 'total_obrigacoes': 88753000.0, 'custo_pessoal_2025_simulado': 284734640.24328, 'plr_inferida': 32511764.71, 'erros': {'F01': 6, 'F02': 9, 'F03': 8, 'F04': 30, 'F05': 12, 'F06': 25}}


## D2. Testes da folha (G01–G06): fictício, duplicado, sem vínculo, maker-checker, alçada e classificação CP/LP

Código-fonte: `codigo/testes_folha.py` (o mesmo código, dividido em células).

In [27]:
# -*- coding: utf-8 -*-
# =============================================================================
# P2 - AUDITORIA 1 - PetroReconcavo (RECV3) - Preparador: matricula 211059370
# >>> SECAO DE SCRIPTS DE AUDITORIA - base complementar de folha (DADOS SINTETICOS) <<<
# Testes em 100% da populacao, executados antes da leitura do log de erros.
# G01 empregado ficticio | G02 CPF/conta duplicados | G03 pagamento sem vinculo (apos desligamento)
# G04 maker-checker no cadastro | G05 reajuste fora de alcada (trilha de alteracoes)
# G06 classificacao circulante x nao circulante (vence ate 31/12/2026 = circulante; a partir de 01/01/2027 = nao circulante)
# Uso: python codigo/testes_folha.py
# =============================================================================
import os
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

F = 'base_simulada/folha'; RES = 'base_simulada/resultados'; os.makedirs(RES, exist_ok=True)
cad = pd.read_csv(f'{F}/CADASTRO_EMPREGADOS_SIMULADO.csv', sep=';', decimal=',', parse_dates=['data_admissao', 'data_desligamento', 'data_ultima_alteracao_salarial'],
                  dtype={'usuario_aprovacao': str}).fillna({'usuario_aprovacao': ''})
fol = pd.read_csv(f'{F}/FOLHA_PAGAMENTOS_2025_SIMULADA.csv', sep=';', decimal=',')
prov = pd.read_csv(f'{F}/OBRIGACOES_TRABALHISTAS_31122025_SIMULADAS.csv', sep=';', decimal=',', parse_dates=['data_vencimento'])
OBRIG_TRAB = 88_753_000.00; DESP_PESSOAL = 284_002_000.00

exc = []
def reg(teste, chaves, detalhe, valor=0.0):
    exc.append(pd.DataFrame({'teste': teste, 'chave': list(chaves), 'detalhe': detalhe, 'valor_em_risco': np.broadcast_to(np.asarray(valor, float), len(chaves))}))
# G01 - pagamentos a matriculas inexistentes no cadastro
g1 = fol[~fol.matricula.isin(cad.matricula)].groupby('matricula').valor_bruto.sum()
reg('G01_EMPREGADO_FICTICIO', g1.index, 'Matricula paga na folha e inexistente no cadastro', g1.values)
# G02 - CPF ou conta bancaria compartilhados por matriculas distintas (marca as matriculas posteriores)
for campo, nome in (('cpf_simulado', 'CPF'), ('banco_agencia_conta', 'conta bancaria')):
    d = cad[cad.duplicated(campo, keep=False)]                        # marca TODAS as matriculas do par (ambas devem ser investigadas)
    reg('G02_CADASTRO_DUPLICADO', d.matricula, f'{nome} igual ao de outra matricula')
# G03 - salario pago em competencia posterior ao desligamento
f = fol.merge(cad[['matricula', 'data_desligamento']], on='matricula', how='inner')
f = f[(f.rubrica == 'SALARIO') & f.data_desligamento.notna() & (f.competencia.str[:7] <= '2025-12')]
f['ini_comp'] = pd.to_datetime(f.competencia + '-01', errors='coerce')
g3 = f[f.ini_comp > f.data_desligamento + pd.offsets.MonthEnd(0)].groupby('matricula').valor_bruto.sum()
reg('G03_PAGAMENTO_SEM_VINCULO', g3.index, 'Salario pago apos o desligamento', g3.values)
# G04 - maker-checker no cadastro
reg('G04_MAKER_CHECKER', cad[(cad.usuario_aprovacao == '') | (cad.usuario_aprovacao == cad.usuario_cadastro)].matricula, 'Cadastro sem aprovacao independente')
# G05 - reajuste > 30% sem aprovacao da diretoria (politica de alcadas simulada)
reg('G05_REAJUSTE_FORA_ALCADA', cad[(cad.pct_ultimo_reajuste > 0.30) & (cad.aprovador_reajuste != 'DIRETORIA_PESSOAS')].matricula, 'Reajuste > 30% aprovado abaixo da alcada')
# G06 - classificacao: vencimento a partir de 01/01/2027 registrado como circulante
g6 = prov[(prov.data_vencimento >= pd.Timestamp('2027-01-01')) & (prov.classificacao_registrada == 'CIRCULANTE')]
reg('G06_CLASSIFICACAO_CP_LP', g6.id_obrigacao, 'Vence a partir de 01/01/2027 e esta no circulante (reclassificar para nao circulante)', g6.valor.values)
excecoes = pd.concat(exc, ignore_index=True)
excecoes.to_csv(f'{RES}/EXCECOES_TESTES_FOLHA.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
resumo = excecoes.groupby('teste').agg(qtd=('chave', 'count'), valor=('valor_em_risco', 'sum')).reset_index()

# conciliacoes
custo = fol.valor_bruto.sum() + fol.encargos_patronais.sum() + fol.beneficios.sum() + prov.loc[prov.tipo_obrigacao == 'PLR_2025_PROVISIONADA', 'valor'].sum() * (1 + 0.288 + 0.08)
conc = pd.DataFrame([
    ('Obrigacoes sociais e trabalhistas 31/12/2025 - base', prov.valor.sum()), ('CVM 2.01.01 consolidado (F03)', OBRIG_TRAB), ('Diferenca', prov.valor.sum() - OBRIG_TRAB),
    ('  dos quais vencem ate 31/12/2026 (circulante)', prov.loc[prov.data_vencimento <= '2026-12-31', 'valor'].sum()),
    ('  dos quais vencem a partir de 01/01/2027 (nao circulante)', prov.loc[prov.data_vencimento >= '2027-01-01', 'valor'].sum()),
    ('Custo de pessoal 2025 - base (bruto + encargos + beneficios + PLR provisionada c/ encargos)', custo), ('Despesa de pessoal NE 19 consolidada (F01 p.93)', DESP_PESSOAL),
    ('Diferenca', custo - DESP_PESSOAL), ('  explicada por G01 + G03 (pagamentos indevidos simulados, com encargos de G03)', np.nan)], columns=['item', 'valor_R$'])

lg = pd.read_csv('base_simulada/controle_erros/LOG_ERROS_INSERIDOS_FOLHA.csv', sep=';', decimal=',')   # lido somente agora
MAPA = {'F01': 'G01_EMPREGADO_FICTICIO', 'F02': 'G02_CADASTRO_DUPLICADO', 'F03': 'G03_PAGAMENTO_SEM_VINCULO', 'F04': 'G04_MAKER_CHECKER',
        'F05': 'G05_REAJUSTE_FORA_ALCADA', 'F06': 'G06_CLASSIFICACAO_CP_LP'}
comp = []
for t, teste in MAPA.items():
    ins, det = set(lg[lg.tipo_erro == t].chave), set(excecoes[excecoes.teste == teste].chave)
    contraparte = set()
    if t == 'F02':                                                    # a outra matricula do par nao e falso positivo: e a contraparte do cadastro duplicado
        alvo = cad[cad.matricula.isin(ins)]
        contraparte = set(cad[(cad.cpf_simulado.isin(alvo.cpf_simulado) | cad.banco_agencia_conta.isin(alvo.banco_agencia_conta)) & ~cad.matricula.isin(ins)].matricula)
    comp.append(dict(tipo_erro=t, teste=teste, inseridos=len(ins), detectados=len(det), verdadeiros_positivos=len(ins & det), contrapartes_do_par=len(det & contraparte),
                     falsos_positivos=len(det - ins - contraparte),
                     taxa_deteccao=len(ins & det) / len(ins), efeito_inserido=lg[lg.tipo_erro == t]['efeito_monetario_R$'].sum()))
comp = pd.DataFrame(comp)
print(resumo); print(conc); print(comp)

HEAD = PatternFill('solid', fgColor='1F4E78'); thin = Side(style='thin', color='BFBFBF'); BD = Border(top=thin, bottom=thin, left=thin, right=thin)
NUM = '#,##0.00;(#,##0.00);-'
def sheet(wb, nome, titulo, df, w, nf=None):
    ws = wb.create_sheet(nome); ws['A1'] = titulo; ws['A1'].font = Font(bold=True, size=13)
    ws['A2'] = 'DADOS SINTETICOS - procedimento SIMULADO PARA FINS DIDATICOS | Preparador: matricula 211059370 | 23/10/2026 | Revisao: ______'
    ws['A2'].font = Font(italic=True, size=9)
    for j, c in enumerate(df.columns, 1):
        x = ws.cell(row=4, column=j, value=c); x.font = Font(bold=True, color='FFFFFF'); x.fill = HEAD; x.border = BD; ws.column_dimensions[get_column_letter(j)].width = w[j - 1]
    for i, row in enumerate(df.itertuples(index=False), 5):
        for j, v in enumerate(row, 1):
            if isinstance(v, float) and np.isnan(v): v = None
            elif isinstance(v, np.generic): v = v.item()
            c = ws.cell(row=i, column=j, value=v); c.border = BD
            if nf and j in nf: c.number_format = nf[j]
wb = Workbook(); wb.remove(wb.active)
sheet(wb, 'Conciliacao', 'Conciliacao da base de folha com a DF (CVM 2.01.01 e NE 19)', conc, [90, 22], {2: NUM})
sheet(wb, 'Resumo_testes', 'Testes G01-G06 em 100% da populacao', resumo, [34, 10, 18], {3: NUM})
sheet(wb, 'Excecoes', 'Excecoes (detalhe)', excecoes, [30, 16, 70, 18], {4: NUM})
sheet(wb, 'Inseridos_x_detectados', 'Comparacao erros inseridos x detectados (folha)', comp, [8, 30, 10, 10, 12, 12, 12, 12, 18], {8: '0.0%', 9: NUM})
wb.save('planilhas/TESTES_FOLHA_RECV3.xlsx'); print('ok')

                       teste  qtd       valor
0     G01_EMPREGADO_FICTICIO    6   680766.66
1     G02_CADASTRO_DUPLICADO   18        0.00
2  G03_PAGAMENTO_SEM_VINCULO    8    37917.87
3          G04_MAKER_CHECKER   30        0.00
4   G05_REAJUSTE_FORA_ALCADA   12        0.00
5    G06_CLASSIFICACAO_CP_LP   25  1887633.53
                                                item      valor_R$
0  Obrigacoes sociais e trabalhistas 31/12/2025 -...  8.875300e+07
1                      CVM 2.01.01 consolidado (F03)  8.875300e+07
2                                          Diferenca  0.000000e+00
3       dos quais vencem ate 31/12/2026 (circulante)  8.686537e+07
4    dos quais vencem a partir de 01/01/2027 (nao...  1.887634e+06
5  Custo de pessoal 2025 - base (bruto + encargos...  2.847346e+08
6    Despesa de pessoal NE 19 consolidada (F01 p.93)  2.840020e+08
7                                          Diferenca  7.326401e+05
8    explicada por G01 + G03 (pagamentos indevido...           NaN
  tipo_e

## E. Dicionário de variáveis (gerado e conferido contra os arquivos)

Código-fonte: `codigo/gerar_dicionario.py` (o mesmo código, dividido em células).

In [28]:
# -*- coding: utf-8 -*-
# P2 - Auditoria 1 - RECV3 - Preparador 211059370
# Gera o DICIONARIO DE VARIAVEIS (xlsx + md) e CONFERE se cada variavel documentada existe nos arquivos gerados.
# Uso: python codigo/gerar_dicionario.py
import os, gzip
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

ARQ = {
 'BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz': ('base_simulada/BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz', 'Base obrigatoria: subrazao analitico do imobilizado em 31/12/2025 - 1 linha = 1 item patrimonial/componente capitalizado'),
 'INSPECAO_DOCUMENTAL_SIMULADA.csv.gz': ('base_simulada/INSPECAO_DOCUMENTAL_SIMULADA.csv.gz', 'Resultado SIMULADO da inspecao de documentos (usado so para itens selecionados na amostra)'),
 'PARAMETROS_CAMPOS_SIMULADOS.csv': ('base_simulada/PARAMETROS_CAMPOS_SIMULADOS.csv', 'Cadastro das 58 concessoes com producao e reservas (rateio sintetico de totais reais)'),
 'PLANO_CONTAS_IMOBILIZADO_SIMULADO.csv': ('base_simulada/PLANO_CONTAS_IMOBILIZADO_SIMULADO.csv', 'Plano de contas analitico SIMULADO do imobilizado'),
 'CADASTRO_POCOS_SIMULADO.csv': ('base_simulada/CADASTRO_POCOS_SIMULADO.csv', 'Cadastro sintetico de pocos'),
 'LOG_ERROS_INSERIDOS.csv': ('base_simulada/controle_erros/LOG_ERROS_INSERIDOS.csv', 'Log de controle dos erros inseridos na base do imobilizado (nao usado pelos testes)'),
 'CADASTRO_EMPREGADOS_SIMULADO.csv': ('base_simulada/folha/CADASTRO_EMPREGADOS_SIMULADO.csv', 'Base complementar: cadastro de empregados'),
 'FOLHA_PAGAMENTOS_2025_SIMULADA.csv': ('base_simulada/folha/FOLHA_PAGAMENTOS_2025_SIMULADA.csv', 'Base complementar: lancamentos da folha de 2025'),
 'OBRIGACOES_TRABALHISTAS_31122025_SIMULADAS.csv': ('base_simulada/folha/OBRIGACOES_TRABALHISTAS_31122025_SIMULADAS.csv', 'Base complementar: obrigacoes trabalhistas em 31/12/2025'),
 'LOG_ERROS_INSERIDOS_FOLHA.csv': ('base_simulada/controle_erros/LOG_ERROS_INSERIDOS_FOLHA.csv', 'Log de controle dos erros inseridos na base de folha'),
}
# (arquivo, variavel, tipo, descricao, dominio / regra de geracao, real ou simulado)
D = [
 ('BASE', 'id_registro', 'texto', 'Identificador unico do registro no subrazao', 'IMB-0000001 a IMB-0532598 (sequencial apos embaralhamento)', 'SIMULADO'),
 ('BASE', 'entidade', 'texto', 'Entidade do grupo titular do item', 'PETRORECONCAVO (controladora) | SPE_TIETA (controlada 100%)', 'Estrutura REAL (NE 1); alocacao SIMULADA'),
 ('BASE', 'polo', 'texto', 'Polo/ativo de producao', 'REMANSO, REMANSO_BT_REC, MIRANGA, POTIGUAR, TIETA, MIDSTREAM_RN', 'Polos REAIS (NE 7.1 ii; FRE 1.1)'),
 ('BASE', 'campo', 'texto', 'Concessao (campo) ou local do item', '58 concessoes (nomes publicos quando disponiveis; "POT-CONC-xx (SINTETICO)" nos demais), blocos e UPGN Guamare', 'Parcialmente REAL'),
 ('BASE', 'uf', 'texto', 'Unidade da federacao', 'BA, SE, RN', 'REAL por polo'),
 ('BASE', 'poco_id', 'texto', 'Poco vinculado (classes desenvolvimento e ARO)', 'PRS-Cnn-nnnn; vazio nas demais classes', 'SIMULADO'),
 ('BASE', 'classe_ne7', 'texto', 'Classe do imobilizado conforme NE 7.1', '9 classes da NE 7.1 (maquinas, andamento, direito de producao, desenvolvimento, blocos, ARO, almoxarifado, adiantamentos, outros)', 'Classes REAIS'),
 ('BASE', 'conta_custo', 'texto', 'Conta analitica de custo (plano simulado)', 'ver PLANO_CONTAS_IMOBILIZADO_SIMULADO.csv', 'SIMULADO'),
 ('BASE', 'descricao_conta', 'texto', 'Descricao da conta de custo', 'texto do plano simulado', 'SIMULADO'),
 ('BASE', 'conta_depreciacao', 'texto', 'Conta redutora de depreciacao/deplecao', '1.2.03.9x.001; vazio para itens que nao depreciam', 'SIMULADO'),
 ('BASE', 'descricao_conta_depreciacao', 'texto', 'Descricao da conta redutora', 'texto', 'SIMULADO'),
 ('BASE', 'tipo_item', 'texto', 'Natureza do componente/item', 'ex.: PERFURACAO E COMPLETACAO; UNIDADE DE BOMBEIO MECANICO; TUBO DE REVESTIMENTO (LOTE)', 'SIMULADO'),
 ('BASE', 'origem', 'texto', 'Origem do registro', 'AQUISICAO_PETROBRAS, AQUISICAO_MAHA, AQUISICAO_BRAVA_JOA, RODADA_ANP, CAPEX_PROPRIO, COMPRA_MATERIAL, ARO_REMENSURACAO', 'Origens REAIS (NE 7, FRE 1.1); alocacao SIMULADA'),
 ('BASE', 'data_aquisicao', 'data', 'Data de aquisicao/capitalizacao/entrada em estoque', 'AAAA-MM-DD; posterior a aquisicao do polo', 'SIMULADO'),
 ('BASE', 'data_inicio_depreciacao', 'data', 'Inicio da depreciacao/deplecao', 'Linear: coerente com a acumulada (acumulada = custo x taxa x anos); UoP: = data de aquisicao', 'SIMULADO'),
 ('BASE', 'metodo_depreciacao', 'texto', 'Metodo de depreciacao', 'LINEAR | UOP_1P | UOP_PDP | NAO_DEPRECIA (NE 7.2)', 'Metodos REAIS'),
 ('BASE', 'taxa_anual_linear', 'decimal', 'Taxa anual (itens lineares)', 'Maquinas 10%; outros 4% a 20% (NE 7.2)', 'Taxas REAIS'),
 ('BASE', 'taxa_uop_campo_2025', 'decimal', 'Taxa de unidades produzidas do campo em 2025', 'producao 2025 / (reservas 31/12/2025 + producao 2025)', 'SIMULADO (a partir de totais reais)'),
 ('BASE', 'custo_historico', 'moeda (R$)', 'Custo historico do item', 'Rateio exato em centavos por entidade x classe; soma = NE 7.1 (arredondamento)', 'Totais REAIS; itens SIMULADOS'),
 ('BASE', 'depreciacao_acumulada', 'moeda (R$)', 'Depreciacao/deplecao acumulada (valor positivo da conta redutora)', 'Soma por entidade x classe = NE 7.1', 'Totais REAIS; itens SIMULADOS'),
 ('BASE', 'valor_contabil_liquido', 'moeda (R$)', 'Custo - depreciacao acumulada', 'soma consolidada = R$ 6.170.841 mil (CVM 1.02.03)', 'Total REAL; itens SIMULADOS'),
 ('BASE', 'depreciacao_2025', 'moeda (R$)', 'Depreciacao/deplecao reconhecida em 2025', 'UoP calibrada a NE 7.1; linear derivada das datas', 'SIMULADO'),
 ('BASE', 'status', 'texto', 'Situacao do item', 'EM_OPERACAO, EM_CONSTRUCAO, EM_ESTOQUE, ADIANTAMENTO, EXPLORACAO, BAIXADO', 'SIMULADO'),
 ('BASE', 'data_baixa', 'data', 'Data da baixa (se baixado)', 'vazio quando nao baixado', 'SIMULADO'),
 ('BASE', 'documento_origem', 'texto', 'Documento que suporta o registro', 'NF-/OI-/CTR-/CCV-/MEM-/ANP- + numero', 'SIMULADO'),
 ('BASE', 'fornecedor_id', 'texto', 'Fornecedor/contraparte', 'FORN-nnnnn; PETROBRAS; VENDEDORES_MAHA; ANP; 3R POTIGUAR (BRAVA); N/A (ESTIMATIVA)', 'SIMULADO (contrapartes reais nas aquisicoes)'),
 ('BASE', 'centro_custo', 'texto', 'Centro de custo', 'CC-<polo>-<classe>', 'SIMULADO'),
 ('BASE', 'garantia_judicial', 'texto', 'Bem dado em garantia judicial', 'S/N (uma sonda = S, NE 7.3)', 'Fato REAL; item SIMULADO'),
 ('BASE', 'usuario_cadastro', 'texto', 'Usuario que cadastrou (maker)', 'USR001-USR028', 'SIMULADO'),
 ('BASE', 'usuario_aprovacao', 'texto', 'Usuario que aprovou (checker)', 'USR001-USR028; deve ser diferente do maker', 'SIMULADO'),
 ('BASE', 'data_cadastro_sistema', 'data', 'Data de inclusao no sistema (trilha)', 'AAAA-MM-DD', 'SIMULADO'),
 ('BASE', 'data_ultima_alteracao', 'data', 'Data da ultima alteracao (trilha de auditoria)', 'AAAA-MM-DD', 'SIMULADO'),
 ('BASE', 'natureza_dado', 'texto', 'Marcador de dado sintetico', 'SINTETICO', 'Marcador'),
 ('INSPECAO', 'id_registro', 'texto', 'Chave para a base', 'igual a base', 'SIMULADO'),
 ('INSPECAO', 'documento_origem', 'texto', 'Documento inspecionado', 'igual a base', 'SIMULADO'),
 ('INSPECAO', 'documento_localizado', 'texto', 'Documento foi localizado?', 'S/N', 'SIMULADO'),
 ('INSPECAO', 'data_documento', 'data', 'Data constante do documento', 'AAAA-MM-DD', 'SIMULADO'),
 ('INSPECAO', 'valor_liquido_suportado', 'moeda (R$)', 'Valor liquido suportado pela evidencia (valor auditado no MUS)', '<= valor contabil', 'SIMULADO'),
 ('INSPECAO', 'observacao_inspecao', 'texto', 'Resultado da inspecao', 'texto', 'SIMULADO'),
 ('INSPECAO', 'natureza_dado', 'texto', 'Marcador', 'SINTETICO', 'Marcador'),
 ('CAMPOS', 'campo', 'texto', 'Nome da concessao', '58 concessoes', 'Parcialmente REAL'),
 ('CAMPOS', 'polo', 'texto', 'Polo', '5 polos', 'REAL'),
 ('CAMPOS', 'entidade', 'texto', 'Titular', 'PETRORECONCAVO | SPE_TIETA', 'REAL'),
 ('CAMPOS', 'ativo', 'texto', 'Ativo (agrupamento do RA)', 'BAHIA | POTIGUAR', 'REAL'),
 ('CAMPOS', 'uf', 'texto', 'UF', 'BA, SE, RN', 'REAL'),
 ('CAMPOS', 'data_aquisicao_polo', 'data', 'Data de aquisicao/inicio do polo', 'FRE 1.1', 'REAL (aproximada)'),
 ('CAMPOS', 'n_pocos', 'inteiro', 'Pocos simulados no campo', 'total 7.200', 'SIMULADO'),
 ('CAMPOS', 'producao_2025_boe', 'inteiro', 'Producao 2025 do campo', 'rateio de 13,4 mil boe/d (Bahia) e 13,1 mil boe/d (Potiguar) - RA', 'Total REAL; rateio SIMULADO'),
 ('CAMPOS', 'reservas_1P_31122025_boe', 'inteiro', 'Reservas provadas 1P', 'rateio de 87,3 MMboe (Bahia) e 56,1 MMboe (Potiguar) - RA/NSAI', 'Total REAL; rateio SIMULADO'),
 ('CAMPOS', 'reservas_PDP_31122025_boe', 'inteiro', 'Reservas PDP', 'rateio proporcional de 61,4 MMboe (RA)', 'Total REAL; rateio SIMULADO'),
 ('CAMPOS', 'taxa_uop_1P_2025', 'decimal', 'Taxa UoP 1P', 'producao/(reserva+producao)', 'SIMULADO'),
 ('CAMPOS', 'taxa_uop_PDP_2025', 'decimal', 'Taxa UoP PDP', 'producao/(reserva+producao)', 'SIMULADO'),
 ('CAMPOS', 'situacao_concessao', 'texto', 'Situacao', 'PRODUCAO', 'REAL (NE 17.1)'),
 ('CAMPOS', 'natureza_dado', 'texto', 'Marcador', 'texto', 'Marcador'),
 ('PLANO', 'conta_custo', 'texto', 'Codigo da conta', '1.2.03.xx.xxx', 'SIMULADO'), ('PLANO', 'descricao_conta', 'texto', 'Descricao', 'texto', 'SIMULADO'),
 ('PLANO', 'classe_ne7', 'texto', 'Classe da NE 7.1', '9 classes', 'REAL'), ('PLANO', 'conta_depreciacao', 'texto', 'Conta redutora', '1.2.03.9x.001', 'SIMULADO'),
 ('PLANO', 'natureza_dado', 'texto', 'Marcador', 'texto', 'Marcador'),
 ('POCOS', 'poco_id', 'texto', 'Identificador do poco', 'PRS-Cnn-nnnn', 'SIMULADO'), ('POCOS', 'campo', 'texto', 'Campo', '58 concessoes', 'Parcialmente REAL'),
 ('POCOS', 'polo', 'texto', 'Polo', '5 polos', 'REAL'), ('POCOS', 'entidade', 'texto', 'Titular', '2 entidades', 'REAL'), ('POCOS', 'uf', 'texto', 'UF', 'BA/SE/RN', 'REAL'),
 ('POCOS', 'data_entrada', 'data', 'Entrada do poco no grupo', 'aquisicao do polo ou perfuracao posterior', 'SIMULADO'),
 ('POCOS', 'origem_poco', 'texto', 'Origem', 'ADQUIRIDO_COM_O_POLO | PERFURADO_PELO_GRUPO', 'SIMULADO'), ('POCOS', 'natureza_dado', 'texto', 'Marcador', 'SINTETICO', 'Marcador'),
 ('LOG', 'seq', 'inteiro', 'Sequencial', '1..n', 'Controle'), ('LOG', 'tipo_erro', 'texto', 'Tipo do erro inserido', 'E01 a E11', 'Controle'),
 ('LOG', 'descricao', 'texto', 'Natureza do erro', 'texto', 'Controle'), ('LOG', 'campo_alterado', 'texto', 'Variavel alterada', 'texto', 'Controle'),
 ('LOG', 'valor_original', 'texto', 'Valor antes da insercao', 'texto', 'Controle'), ('LOG', 'valor_inserido', 'texto', 'Valor inserido', 'texto', 'Controle'),
 ('LOG', 'id_registro', 'texto', 'Registro afetado', 'IMB-...', 'Controle'), ('LOG', 'efeito_monetario_R$', 'moeda (R$)', 'Efeito no valor liquido (+ superavaliacao; - subavaliacao; 0 = nao monetario)', 'R$', 'Controle'),
 ('EMP', 'matricula', 'texto', 'Matricula do empregado', 'M00001...', 'SIMULADO'), ('EMP', 'entidade', 'texto', 'Empregador', '2 entidades', 'SIMULADO'),
 ('EMP', 'cpf_simulado', 'texto', 'CPF ficticio (nao e CPF real)', 'SIM-nnnnnnnnn', 'SIMULADO'), ('EMP', 'nivel', 'texto', 'Nivel hierarquico', 'LIDERANCA (249) | NAO_LIDERANCA (1.503) entre os ativos', 'Quantidades REAIS (FRE 10.1A)'),
 ('EMP', 'lotacao', 'texto', 'Lotacao', 'sede e polos', 'SIMULADO'), ('EMP', 'data_admissao', 'data', 'Admissao', 'AAAA-MM-DD', 'SIMULADO'),
 ('EMP', 'data_desligamento', 'data', 'Desligamento (vazio = ativo em 31/12/2025)', 'AAAA-MM-DD', 'SIMULADO'), ('EMP', 'salario_base_mensal', 'moeda (R$)', 'Salario base', 'calibrado a despesa de pessoal da NE 19', 'SIMULADO'),
 ('EMP', 'banco_agencia_conta', 'texto', 'Conta bancaria ficticia', 'bbb-aaaa-ccccccc', 'SIMULADO'), ('EMP', 'contrato_trabalho_arquivado', 'texto', 'Contrato arquivado', 'S/N', 'SIMULADO'),
 ('EMP', 'usuario_cadastro', 'texto', 'Maker (RH)', 'RH01-RH08', 'SIMULADO'), ('EMP', 'usuario_aprovacao', 'texto', 'Checker (RH)', 'RH01-RH08, diferente do maker', 'SIMULADO'),
 ('EMP', 'data_ultima_alteracao_salarial', 'data', 'Trilha: ultima alteracao salarial', 'AAAA-MM-DD', 'SIMULADO'), ('EMP', 'pct_ultimo_reajuste', 'decimal', 'Ultimo reajuste (%)', '0 a 65%', 'SIMULADO'),
 ('EMP', 'aprovador_reajuste', 'texto', 'Alcada que aprovou o reajuste', 'GERENCIA_RH | DIRETORIA_PESSOAS (acima de 30% exige diretoria)', 'SIMULADO'), ('EMP', 'natureza_dado', 'texto', 'Marcador', 'SINTETICO', 'Marcador'),
 ('FOL', 'id_lancamento', 'texto', 'Id do lancamento', 'FOL-nnnnnnn', 'SIMULADO'), ('FOL', 'competencia', 'texto', 'Competencia', '2025-01 a 2025-12; 2025-13 = 13o', 'SIMULADO'),
 ('FOL', 'matricula', 'texto', 'Matricula paga', 'chave do cadastro', 'SIMULADO'), ('FOL', 'rubrica', 'texto', 'Rubrica', 'SALARIO, ADICIONAL_PERICULOSIDADE, HORAS_EXTRAS, 13O_SALARIO, FERIAS_+1/3, PLR_2024_PAGA, VERBAS_RESCISORIAS', 'SIMULADO'),
 ('FOL', 'valor_bruto', 'moeda (R$)', 'Valor bruto', 'R$', 'SIMULADO'), ('FOL', 'data_pagamento', 'data', 'Data do pagamento', 'AAAA-MM-DD', 'SIMULADO'),
 ('FOL', 'conta_credito', 'texto', 'Conta creditada', 'conta do cadastro', 'SIMULADO'), ('FOL', 'encargos_patronais', 'moeda (R$)', 'INSS patronal + FGTS (36,8%)', 'R$', 'SIMULADO'),
 ('FOL', 'beneficios', 'moeda (R$)', 'Beneficios (11,5% da folha bruta)', 'R$', 'SIMULADO'), ('FOL', 'natureza_dado', 'texto', 'Marcador', 'SINTETICO', 'Marcador'),
 ('OBR', 'id_obrigacao', 'texto', 'Id da obrigacao', 'PROV-nnnnnn', 'SIMULADO'), ('OBR', 'matricula', 'texto', 'Empregado', 'chave do cadastro', 'SIMULADO'),
 ('OBR', 'tipo_obrigacao', 'texto', 'Natureza', 'salario de dezembro, encargos, ferias, PLR 2025, outras, bonus de retencao', 'SIMULADO'),
 ('OBR', 'valor', 'moeda (R$)', 'Saldo em 31/12/2025', 'soma = R$ 88.753 mil (CVM 2.01.01)', 'Total REAL; itens SIMULADOS'),
 ('OBR', 'data_vencimento', 'data', 'Vencimento', 'ate 31/12/2026 = circulante; a partir de 01/01/2027 = nao circulante', 'SIMULADO'),
 ('OBR', 'classificacao_registrada', 'texto', 'Classificacao no balanco', 'CIRCULANTE', 'SIMULADO'), ('OBR', 'natureza_dado', 'texto', 'Marcador', 'SINTETICO', 'Marcador'),
 ('LOGF', 'tipo_erro', 'texto', 'Tipo', 'F01 a F06', 'Controle'), ('LOGF', 'chave', 'texto', 'Matricula ou id_obrigacao afetado', 'texto', 'Controle'),
 ('LOGF', 'descricao', 'texto', 'Natureza do erro', 'texto', 'Controle'), ('LOGF', 'efeito_monetario_R$', 'moeda (R$)', 'Efeito monetario', 'R$', 'Controle'),
]
SIG = {'BASE': 'BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz', 'INSPECAO': 'INSPECAO_DOCUMENTAL_SIMULADA.csv.gz', 'CAMPOS': 'PARAMETROS_CAMPOS_SIMULADOS.csv',
       'PLANO': 'PLANO_CONTAS_IMOBILIZADO_SIMULADO.csv', 'POCOS': 'CADASTRO_POCOS_SIMULADO.csv', 'LOG': 'LOG_ERROS_INSERIDOS.csv', 'EMP': 'CADASTRO_EMPREGADOS_SIMULADO.csv',
       'FOL': 'FOLHA_PAGAMENTOS_2025_SIMULADA.csv', 'OBR': 'OBRIGACOES_TRABALHISTAS_31122025_SIMULADAS.csv', 'LOGF': 'LOG_ERROS_INSERIDOS_FOLHA.csv'}
dic = pd.DataFrame(D, columns=['arquivo', 'variavel', 'tipo', 'descricao', 'dominio_regra', 'real_ou_simulado'])
dic['arquivo'] = dic.arquivo.map(SIG)
# conferencia: variaveis documentadas x colunas reais dos arquivos
prob = []; linhas = {}
for nome, (path, desc) in ARQ.items():
    df = pd.read_csv(path, sep=';', nrows=5, encoding='utf-8-sig' if not path.endswith('.gz') else 'utf-8')
    doc = set(dic[dic.arquivo == nome].variavel); real = set(df.columns)
    if doc != real: prob.append((nome, sorted(real - doc), sorted(doc - real)))
    with (gzip.open(path, 'rt', encoding='utf-8') if path.endswith('.gz') else open(path, encoding='utf-8-sig')) as f:
        linhas[nome] = sum(1 for _ in f) - 1
assert not prob, prob
wb = Workbook(); ws = wb.active; ws.title = 'Arquivos'
ws.append(['Arquivo', 'Caminho', 'Conteudo', 'Linhas']); [ws.append([k, v[0], v[1], linhas[k]]) for k, v in ARQ.items()]
ws2 = wb.create_sheet('Variaveis'); ws2.append(list(dic.columns)); [ws2.append(list(r)) for r in dic.itertuples(index=False)]
for w, larg in ((ws, [44, 62, 90, 10]), (ws2, [44, 30, 12, 60, 80, 34])):
    for j, x in enumerate(larg, 1): w.column_dimensions[get_column_letter(j)].width = x
    for c in w[1]: c.font = Font(bold=True, color='FFFFFF'); c.fill = PatternFill('solid', fgColor='1F4E78')
    for row in w.iter_rows(min_row=2):
        for c in row: c.alignment = Alignment(wrap_text=True, vertical='top')
wb.save('base_simulada/DICIONARIO_VARIAVEIS.xlsx')
md = ['# Dicionario de variaveis - bases simuladas (DADOS SINTETICOS)', '', 'Preparador: matricula 211059370. Gerado por `codigo/gerar_dicionario.py`, que tambem confere se cada variavel documentada existe no arquivo.', '',
      '## Arquivos', '', '| Arquivo | Linhas | Conteudo |', '|---|---:|---|'] + [f'| `{v[0]}` | {linhas[k]:,} | {v[1]} |'.replace(',', '.') for k, v in ARQ.items()]
for nome in ARQ:
    md += ['', f'## {nome}', '', '| Variavel | Tipo | Descricao | Dominio / regra | Real ou simulado |', '|---|---|---|---|---|']
    md += [f'| `{r.variavel}` | {r.tipo} | {r.descricao} | {r.dominio_regra} | {r.real_ou_simulado} |' for r in dic[dic.arquivo == nome].itertuples()]
open('base_simulada/DICIONARIO_VARIAVEIS.md', 'w', encoding='utf-8').write('\n'.join(md) + '\n')
print('ok dicionario', {k: linhas[k] for k in linhas})

ok dicionario {'BASE_IMOBILIZADO_RECV3_SINTETICA.csv.gz': 532598, 'INSPECAO_DOCUMENTAL_SIMULADA.csv.gz': 532598, 'PARAMETROS_CAMPOS_SIMULADOS.csv': 58, 'PLANO_CONTAS_IMOBILIZADO_SIMULADO.csv': 29, 'CADASTRO_POCOS_SIMULADO.csv': 7200, 'LOG_ERROS_INSERIDOS.csv': 1301, 'CADASTRO_EMPREGADOS_SIMULADO.csv': 1942, 'FOLHA_PAGAMENTOS_2025_SIMULADA.csv': 61426, 'OBRIGACOES_TRABALHISTAS_31122025_SIMULADAS.csv': 9160, 'LOG_ERROS_INSERIDOS_FOLHA.csv': 90}


## Conclusão do passo 3 (resumo, dados sintéticos)
Os números finais estão em `base_simulada/resultados/` e nas planilhas `CONC_BASE_RECV3.xlsx`, `TESTES_BASE_RECV3.xlsx` e `TESTES_FOLHA_RECV3.xlsx`. A interpretação vai nos PTs 500, 530 e 450 (passo 4). Nenhum resultado deste notebook é uma constatação sobre a PetroReconcavo real.